## **Step 2: Hallucination Control & Ablation Evaluation**

**Project:** *Towards Explainable and Reliable Large Language Models for Clinical Decision Support*

### Notebook objective

Detect hallucinations at the **claim level**, mitigate unsupported or contradicted claims using verified biomedical evidence, and evaluate the contribution of the **Hallucination Control Module** through an ablation study comparing the pipeline **without** and **with** hallucination control on the same 100 PubMedQA questions.

### Final selected configuration (V5)

| Component                         | Final choice                           |
| --------------------------------- | -------------------------------------- |
| Dataset                           | PubMedQA                               |
| Evaluation sample                 | Same fixed 100 questions               |
| Claim decomposition model         | `gemini-flash-lite-latest`             |
| Gemini temperature                | `0.0`                                  |
| NLI model                         | `microsoft/deberta-large-mnli`         |
| NLI entailment threshold          | `0.60`                                 |
| NLI contradiction threshold       | `0.60`                                 |
| Biomedical relevance model        | `pritamdeka/S-PubMedBert-MS-MARCO`     |
| Biomedical similarity threshold   | `0.30`                                 |
| Maximum mitigation attempts       | `4`                                    |
| Re-retrieval method               | MiniLM + PubMedBERT                    |
| Vector database                   | Qdrant                                 |
| RRF fusion                        | `RRF_K = 60`                           |
| Retrieved candidates per index    | `20`                                   |
| Kept re-retrieved evidence        | `8`                                    |
| Cross-encoder reranker            | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Maximum correction attempts       | `2`                                    |
| Maximum grounding attempts        | `3`                                    |
| Correction entailment threshold   | `0.50`                                 |
| Maximum contradiction probability | `0.30`                                 |
| Minimum lexical overlap           | `0.25`                                 |
| Numeric fidelity verification     | Enabled                                |
| Grounding fallback                | Enabled                                |
| Soft-accept for NEE claims        | Enabled (low-contradiction fallback)   |
| BERTScore-F1 model                | `allenai/scibert_scivocab_uncased`     |
| Checkpoint / resume               | Enabled                                |

### Main pipeline steps

1. Load the same **100 generated answers and retrieved evidence** produced by Notebook 1.
2. Decompose each generated answer into **atomic medical claims** using Gemini Flash-Lite.
3. Verify each claim against its retrieved evidence using DeBERTa NLI and biomedical semantic similarity.
4. Classify each claim as **SUPPORTED**, **CONTRADICTED**, or **NOT_ENOUGH_EVIDENCE**.
5. Keep `SUPPORTED` claims unchanged.
6. Rewrite `CONTRADICTED` claims using verified evidence and re-verify them with NLI before acceptance.
7. For `NOT_ENOUGH_EVIDENCE`, generate a targeted retrieval query and perform hybrid re-retrieval using MiniLM + PubMedBERT, RRF fusion, and cross-encoder reranking.
8. Re-verify the claim against the new evidence and keep or correct it when sufficient evidence is found.
9. If the claim remains unresolved, apply an evidence-grounded rewrite and verify it again.
10. Escalate claims that remain unresolved to **human expert review** instead of emitting unsupported information.
11. Reconstruct the final controlled answers using only accepted claims.
12. Perform the ablation evaluation on the **same 100 questions**: without Hallucination Control vs with Hallucination Control.
13. Save claim verification, mitigation traces, human-review cases, controlled answers, and final ablation results.

### Hallucination detection results — Before control

| Metric                 |           Result |
| ---------------------- | ---------------: |
| Total claims           |          **644** |
| Supported claims       | **465 (72.20%)** |
| Contradicted claims    |   **23 (3.57%)** |
| Not enough evidence    | **156 (24.22%)** |
| Unsupported claim rate |       **27.80%** |

### Hallucination mitigation results

| Mitigation outcome                         |  Claims |
| ------------------------------------------ | ------: |
| Originally supported and kept              | **465** |
| Corrected                                  |  **13** |
| Evidence-grounded rewrite                  |  **83** |
| Resolved after re-retrieval                |   **9** |
| Corrected after re-retrieval               |   **5** |
| Escalated to human review                  |  **69** |
| Automatically recovered problematic claims | **110** |

### Final ablation results

| Metric                               | Without Hallucination Control | With Hallucination Control |
| ------------------------------------ | ----------------------------: | -------------------------: |
| Questions evaluated                  |                       **100** |                    **100** |
| Supported claim rate                 |                    **72.20%** |                 **99.65%** |
| Contradicted claim rate              |                     **3.57%** |                  **0.00%** |
| Unsupported claim rate               |                    **27.80%** |                  **0.35%** |
| Faithfulness                         |                    **0.8120** |                 **0.9879** |
| FActScore                            |                    **0.7220** |                 **0.9965** |
| Hallucination Rate (HR)              |                    **27.80%** |                  **0.35%** |
| Answer-level Hallucination Rate      |                    **44.00%** |                  **2.00%** |
| Claims automatically recovered       |                             — |                    **110** |
| Claims escalated to human review     |                             — |                     **69** |


---
# 1. Setup and Configuration

In [1]:
# Section 1 - Setup and Configuration
import ast
import json
import os
import random
import re
import string
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from google import genai
from google.genai import types
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import nltk
from nltk.tokenize import sent_tokenize
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
try:
    from nltk.translate.meteor_score import meteor_score
    HAS_METEOR = True
except Exception:
    meteor_score = None
    HAS_METEOR = False

from IPython.display import display, Markdown

# Resolve .env explicitly (find_dotenv uses the caller's file dir, which is
# unreliable inside a notebook); search cwd and its parents a few levels up.
from dotenv import load_dotenv
_env_candidates = [Path.cwd() / ".env"] + [p / ".env" for p in Path.cwd().parents[:4]]
_env_path = next((p for p in _env_candidates if p.exists()), None)
if _env_path is not None:
    load_dotenv(_env_path, override=False)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Please make sure a .env file with GEMINI_API_KEY=... exists "
        "in the notebook working directory."
    )
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "data" / "outputs"
PIPELINE_OUTPUTS_DIR = OUTPUTS_DIR / "final_rag_pipeline_v5"

HALL_CONTROL_DIR = OUTPUTS_DIR / "hallucination_control"
HALL_CONTROL_DIR.mkdir(parents=True, exist_ok=True)

# Inputs: Notebook 1 outputs (1000 generated answers) + the 100 selected samples.
GENERATION_CSV_PATH = PIPELINE_OUTPUTS_DIR / "generation_results_final_temp_0.2.csv"
SELECTED_CSV_PATH = PIPELINE_OUTPUTS_DIR / "final_selected_100_samples.csv"

# Canonical 100-question baseline used for the ablation.
BASELINE_100_CSV_PATH = HALL_CONTROL_DIR / "baseline_100_questions.csv"

# Outputs (Section 4 - claim decomposition, checkpointed for resume)
CLAIM_DECOMPOSITION_CHECKPOINT_PATH = HALL_CONTROL_DIR / "claim_decomposition_checkpoint.jsonl"

# Outputs (Section 5 - detection). The JSONL doubles as a checkpoint for resume.
CLAIM_VERIFICATION_CSV_PATH = HALL_CONTROL_DIR / "claim_verification.csv"
CLAIM_VERIFICATION_JSONL_PATH = HALL_CONTROL_DIR / "claim_verification.jsonl"
ANSWER_SUMMARY_CSV_PATH = HALL_CONTROL_DIR / "answer_summary.csv"
ANSWER_SUMMARY_JSONL_PATH = HALL_CONTROL_DIR / "answer_summary.jsonl"
NLI_DIAGNOSTICS_CSV_PATH = HALL_CONTROL_DIR / "nli_diagnostics.csv"

# Outputs (Sections 6-7 - mitigation). The JSONL doubles as a checkpoint for resume.
MITIGATION_TRACE_CSV_PATH = HALL_CONTROL_DIR / "mitigation_trace.csv"
MITIGATION_TRACE_JSONL_PATH = HALL_CONTROL_DIR / "mitigation_trace.jsonl"
MITIGATED_ANSWERS_CSV_PATH = HALL_CONTROL_DIR / "mitigated_rag_answers.csv"
MITIGATED_ANSWERS_JSONL_PATH = HALL_CONTROL_DIR / "mitigated_rag_answers.jsonl"
HUMAN_REVIEW_LOG_CSV_PATH = HALL_CONTROL_DIR / "human_expert_review_log.csv"
HUMAN_REVIEW_LOG_JSONL_PATH = HALL_CONTROL_DIR / "human_expert_review_log.jsonl"

# Outputs (Sections 8-10 - ablation)
ABLATION_CSV_PATH = HALL_CONTROL_DIR / "ablation_comparison.csv"
MITIGATION_OUTCOME_CSV_PATH = HALL_CONTROL_DIR / "mitigation_outcome.csv"
BASELINE_PER_Q_METRICS_CSV = HALL_CONTROL_DIR / "ablation_baseline_per_question_metrics.csv"
CONTROLLED_PER_Q_METRICS_CSV = HALL_CONTROL_DIR / "ablation_controlled_per_question_metrics.csv"

# ---------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------
GEMINI_MODEL_NAME = "gemini-flash-lite-latest"
GEMINI_TEMPERATURE = 0.0

# Robust Gemini retries (429 quota / 5xx): exponential backoff with jitter.
GEMINI_MAX_RETRIES = 6
GEMINI_BACKOFF_BASE_SECONDS = 2.0
GEMINI_MAX_BACKOFF_SECONDS = 90.0

# NLI backbone for claim verification.
NLI_MODEL_NAME = "microsoft/deberta-large-mnli"

EMBEDDING_MODEL_MINILM = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_PUBMEDBERT = "pritamdeka/S-PubMedBert-MS-MARCO"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

COLLECTION_MINILM = "pubmedqa_chunks"
COLLECTION_PUBMEDBERT = "pubmedqa_chunks_pubmedbert"

# BERTScore configuration - IDENTICAL to Notebook 1, used for both ablation conditions.
BERTSCORE_MODEL = "allenai/scibert_scivocab_uncased"

# ---------------------------------------------------------------------
# NLI decision thresholds (strict acceptance, reduces false positives)
# ---------------------------------------------------------------------
NLI_ENTAILMENT_THRESHOLD = 0.60
NLI_CONTRADICTION_THRESHOLD = 0.60

# Biomedical relevance gate: an entailment is only accepted when the claim and the
# supporting chunk are also similar under a biomedical encoder.
USE_BIOMEDICAL_RELEVANCE_GATE = True
MIN_CLAIM_EVIDENCE_SIMILARITY = 0.30

# ---------------------------------------------------------------------
# Mitigation controls (bounded retry loop + evidence-derived rewrite policy)
# ---------------------------------------------------------------------
MAX_MITIGATION_ATTEMPTS = 4
# Re-retrieval now searches with TWO query variants (a Gemini query and the raw
# claim) per index, and keeps a larger pool so NLI can find a supporting chunk
# even when the cross-encoder ranks it slightly lower.
RERETRIEVE_TOP_K_PER_INDEX = 20
RERETRIEVE_KEEP_TOP_N = 8
# The ms-marco cross-encoder outputs logits that are mostly negative on biomedical
# pairs, so an absolute cutoff of 0.0 drops almost every re-retrieved chunk. The
# cross-encoder is used for RANKING only; support is decided by NLI afterwards.
MIN_RERETRIEVAL_RELEVANCE = -999.0
RRF_K = 60

# Evidence-derived rewrite acceptance policy (verify-before-accept for CORRECTED
# and GROUNDED claims). A rewrite is only emitted if at least one evidence chunk
# satisfies ALL of: entailment >= CORRECTION_ACCEPT_ENTAILMENT, contradiction
# probability < CORRECTION_ACCEPT_MAX_CONTRADICTION, the biomedical relevance gate,
# every numeric value of the rewrite present in that chunk (numeric fidelity), and
# a minimum lexical overlap. Detection thresholds (NLI_*_THRESHOLD above) are NOT
# changed - this is a stricter-union gate used only for LLM-produced rewrites.
MAX_CORRECTION_ATTEMPTS = 2
MAX_GROUNDING_ATTEMPTS = 3
CORRECTION_ACCEPT_ENTAILMENT = 0.50
CORRECTION_ACCEPT_MAX_CONTRADICTION = 0.30
MIN_CORRECTION_LEXICAL_OVERLAP = 0.25
USE_GROUNDING_FALLBACK = True
GROUNDING_CONTEXT_CHAR_CAP = 4000

# Soft-accept fallback for NEE claims: when all mitigation attempts fail, keep
# the original claim instead of escalating IF it does not contradict any evidence
# (best contradiction prob < threshold) and has decent biomedical relevance.
USE_SOFT_ACCEPT_NEE = True
SOFT_ACCEPT_MAX_CONTRADICTION_PROB = 0.40
SOFT_ACCEPT_MIN_BIO_SIMILARITY = 0.30

# ---------------------------------------------------------------------
# Notebook 1 sentence-level faithfulness constants (identical implementation)
# ---------------------------------------------------------------------
ENTAIL_WEIGHT = 0.60
OVERLAP_WEIGHT = 0.40
SUPPORTED_COMBINED_THRESHOLD = 0.55
CONTRADICT_PROB_THRESHOLD = 0.55

# Ablation must compare the exact same number of questions in both conditions.
REQUIRED_N = 100

# ---------------------------------------------------------------------
# Label maps
# ---------------------------------------------------------------------
NLI_LABEL_TO_VERIFICATION = {
    "entailment": "SUPPORTED",
    "contradiction": "CONTRADICTED",
    "neutral": "NOT_ENOUGH_EVIDENCE",
}
VERIFICATION_LABEL_TO_ACTION = {
    "SUPPORTED": "keep",
    "CONTRADICTED": "correct",
    "NOT_ENOUGH_EVIDENCE": "re_retrieve",
}
INSUFFICIENT_EVIDENCE_MARKER = "insufficient verified evidence"
CORRECTION_FAILED_TEXT = "This claim could not be corrected automatically due to an API error."

# ---------------------------------------------------------------------
# Small checkpoint helpers (JSONL append + load)
# ---------------------------------------------------------------------
def _load_jsonl(path):
    """Load a JSONL file. Returns a list of dicts (missing file -> empty list)."""
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def _append_jsonl(path, record):
    """Atomically append one record to a JSONL checkpoint file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


print("Configuration loaded.")


Configuration loaded.


---
# 2. Load the Same 100 Questions

In [2]:
# Section 2 - Load the same 100 questions (baseline condition).
if not GENERATION_CSV_PATH.exists():
    raise FileNotFoundError(f"{GENERATION_CSV_PATH} not found. Run Notebook 1 first.")
if not SELECTED_CSV_PATH.exists():
    raise FileNotFoundError(f"{SELECTED_CSV_PATH} not found.")

generation_df = pd.read_csv(GENERATION_CSV_PATH)
selected_df = pd.read_csv(SELECTED_CSV_PATH)

df = generation_df.merge(selected_df[["question", "id"]], on="question", how="inner")
df = df.drop_duplicates(subset="question", keep="first").reset_index(drop=True)

if len(df) != REQUIRED_N:
    raise RuntimeError(
        f"Expected exactly {REQUIRED_N} merged samples but got {len(df)}. "
        "The ablation would be unfair - fix the input files before continuing."
    )


def _parse_retrieved_chunks(value):
    """Parse a kept-chunks column (stored as JSON or Python-repr string)."""
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            try:
                value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []
    if not isinstance(value, list):
        return []
    cleaned = []
    for chunk in value:
        if not isinstance(chunk, dict):
            continue
        cleaned.append({
            "chunk_id": chunk.get("chunk_id"),
            "pmid": chunk.get("pmid"),
            "chunk_text": chunk.get("chunk_text"),
            "score": chunk.get("score"),
            "rank": chunk.get("rank"),
            "source": chunk.get("source", "PubMedQA"),
        })
    return cleaned


df["kept_chunks_parsed"] = df["kept_chunks"].apply(_parse_retrieved_chunks)
df["answer_text"] = (
    df["cleaned_answer_for_metrics"]
    .fillna(df["raw_answer"])
    .fillna(df["original_cited_answer"])
)
df["gold_pmid"] = df["gold_pmid"].fillna("").astype(str).str.strip()

# question_id = DataFrame index (0..99), used consistently in every log.
df = df.reset_index(drop=True)

# Save the canonical 100-question baseline for full traceability of the ablation.
df[[
    "question", "id", "gold_pmid", "gold_long_answer", "gold_final_decision",
    "answer_text", "raw_answer", "original_cited_answer", "cleaned_answer_for_metrics",
]].to_csv(BASELINE_100_CSV_PATH, index=False)
print(f"Saved: {BASELINE_100_CSV_PATH}")

n_with_evidence = (df["kept_chunks_parsed"].apply(len) > 0).sum()
print(f"Rows with at least one evidence chunk: {n_with_evidence} / {len(df)}")
print(f"answer_text missing values: {int(df['answer_text'].isna().sum())}")

print("\nExample (first row):")
print(f"Question   : {df.loc[0, 'question']}")
print(f"Answer     : {str(df.loc[0, 'answer_text'])[:220]}...")
print(f"Kept chunks: {len(df.loc[0, 'kept_chunks_parsed'])}")


Saved: d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\baseline_100_questions.csv
Rows with at least one evidence chunk: 100 / 100
answer_text missing values: 0

Example (first row):
Question   : Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?
Answer     : Rapid prescreening (RPS) is a quality assurance method used in gynecologic cytology. A total of 80,565 Papanicolaou smears underwent RPS during a 25-month period. The final cytologic interpretation was a glandular cell a...
Kept chunks: 1


---
# 3. Load Generated Answers and Evidence

In [3]:
# Section 3 - Evidence available per question.
ev_counts = df["kept_chunks_parsed"].apply(len)
print("Kept evidence chunks per question:")
print(ev_counts.describe().to_string())

# Chunk IDs already used by each answer -> excluded from re-retrieval during mitigation.
KEPT_CHUNK_IDS_BY_QUESTION = {
    qid: {c.get("chunk_id") for c in chunks}
    for qid, chunks in df["kept_chunks_parsed"].items()
}

# Quick sanity print
for qid in list(df.index[:5]):
    chunks = df.loc[qid, "kept_chunks_parsed"]
    print(f"  q{qid}: {len(chunks)} kept chunks | first: {str(chunks[0]['chunk_text'])[:90] if chunks else '-'}")


Kept evidence chunks per question:
count    100.000000
mean       1.430000
std        0.623691
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        3.000000
  q0: 1 kept chunks | first: Rapid prescreening (RPS) is one of the quality assurance (QA) methods used in gynecologic 
  q1: 1 kept chunks | first: It is generally assumed, that patients with Werlhof's disease (WD) are at increased risk f
  q2: 2 kept chunks | first: Power calculation stipulated a study population of 323 patients. An association between su
  q3: 1 kept chunks | first: Broad-based electronic health information exchange (HIE), in which patients' clinical data
  q4: 1 kept chunks | first: Specialty pharmaceuticals have evolved beyond their status as niche drugs designed to trea


---
# 4. Claim Decomposition 

In [4]:
# Section 4 - Claim decomposition (Gemini Flash-Lite)
# retries + checkpoint + resume.

# ---------------------------------------------------------------------
# Gemini generation configuration
# ---------------------------------------------------------------------
# The Gemini client itself was created once in Section 1.
# This cell only defines the JSON generation configuration and the
# claim-decomposition logic.

gemini_generation_config = types.GenerateContentConfig(
    temperature=GEMINI_TEMPERATURE,
    response_mime_type="application/json",
)


# ---------------------------------------------------------------------
# Robust Gemini call
# ---------------------------------------------------------------------

def call_gemini_robust(
    prompt,
    max_retries=GEMINI_MAX_RETRIES,
    base_delay=GEMINI_BACKOFF_BASE_SECONDS,
):
    """
    Call Gemini with exponential backoff and jitter.

    The same helper is reused later by the mitigation module.

    Returns
    -------
    response
        Gemini response object if successful.

    None
        If all attempts fail.
    """

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=prompt,
                config=gemini_generation_config,
            )

            return response

        except Exception as exc:
            last_error = exc
            error_text = str(exc)

            is_quota_error = (
                "429" in error_text
                or "quota" in error_text.lower()
                or "resource_exhausted" in error_text.lower()
                or "rate limit" in error_text.lower()
            )

            delay = min(
                GEMINI_MAX_BACKOFF_SECONDS,
                base_delay * (2 ** (attempt - 1))
                + random.uniform(0.0, 1.0),
            )

            # Give quota/rate-limit errors more time before retrying.
            if is_quota_error:
                delay = min(
                    GEMINI_MAX_BACKOFF_SECONDS,
                    delay * 2.0,
                )

            print(
                f"    [Gemini] attempt {attempt}/{max_retries} failed "
                f"({type(exc).__name__}): "
                f"{error_text[:140]} "
                f"-> retrying in {delay:.1f}s"
            )

            time.sleep(delay)

    print(
        f"  [Gemini] call failed after "
        f"{max_retries} attempts: {last_error}"
    )

    return None


# ---------------------------------------------------------------------
# Atomic claim-decomposition prompt
# ---------------------------------------------------------------------

CLAIM_DECOMPOSITION_PROMPT_TEMPLATE = "\n".join([
    "You are a medical claim extraction assistant.",
    "Split the following medical answer into atomic factual medical claims.",
    "",
    "Requirements:",
    "- Each claim must contain exactly ONE medical fact.",
    "- Preserve numbers, percentages, drug names, doses, dates and negations exactly as stated.",
    "- Do not split a single fact into several claims.",
    "- Do not merge several independent facts into one claim.",
    "- Do not include citations, evidence markers, or disclaimers.",
    "- Do not add or infer information that is not stated in the answer.",
    "- Preserve the original medical meaning.",
    "- Return a JSON list of strings and nothing else.",
    "",
    "Answer:",
    "{answer}",
])


# ---------------------------------------------------------------------
# Extract JSON list returned by Gemini
# ---------------------------------------------------------------------

def _extract_json_list(text):

    if not text or not text.strip():
        return None

    text = text.strip()

    # Defensive removal of Markdown fences.
    if text.startswith("```"):

        text = re.sub(
            r"^```[a-zA-Z]*\s*",
            "",
            text,
        )

        text = re.sub(
            r"\s*```\s*$",
            "",
            text,
        )

        text = text.strip()

    # Expected case: Gemini returns a JSON list directly.
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return parsed

    except Exception:
        pass

    # Fallback if extra text surrounds the JSON array.
    match = re.search(
        r"\[.*\]",
        text,
        re.DOTALL,
    )

    if match:

        try:
            parsed = json.loads(
                match.group(0)
            )

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return None


# ---------------------------------------------------------------------
# Decompose one generated answer
# ---------------------------------------------------------------------

def decompose_answer_into_claims(
    answer_text,
):
    """
    Decompose one generated medical answer into atomic factual claims.

    Returns
    -------
    list
        Successful decomposition.

    None
        Gemini/API/parsing failure.

    Important:
    None is deliberately different from [].
    A failed API request must never be interpreted as an answer containing
    zero claims.
    """

    prompt = CLAIM_DECOMPOSITION_PROMPT_TEMPLATE.format(
        answer=str(answer_text)
    )

    response = call_gemini_robust(
        prompt
    )

    if response is None:
        return None

    response_text = (
        response.text
        if response.text is not None
        else ""
    )

    parsed = _extract_json_list(
        response_text
    )

    if parsed is None:

        print(
            "  [Gemini] claim decomposition "
            "returned unparseable JSON."
        )

        return None

    claims = []

    for claim in parsed:

        if not isinstance(
            claim,
            (str, int, float),
        ):
            continue

        claim = str(
            claim
        ).strip()

        if not claim:
            continue

        # Defensive removal of evidence markers.
        claim = re.sub(
            r"\(\s*Evidence\s*\d+\s*\)",
            "",
            claim,
            flags=re.IGNORECASE,
        )

        claim = re.sub(
            r"\s+",
            " ",
            claim,
        ).strip()

        if claim:
            claims.append(
                claim
            )

    # Remove exact duplicate claims while preserving order.
    claims = list(
        dict.fromkeys(
            claims
        )
    )

    return claims


# ---------------------------------------------------------------------
# Resume already completed claim decomposition
# ---------------------------------------------------------------------

all_claims = {}


# First recover claims already used by the verification stage.
for rec in _load_jsonl(
    CLAIM_VERIFICATION_JSONL_PATH
):

    qid = rec["question_id"]

    claim_id = str(
        rec.get(
            "claim_id",
            "",
        )
    )

    suffix = claim_id.rsplit(
        "_",
        1,
    )[-1]

    claim_index = (
        int(suffix)
        if suffix.isdigit()
        else 0
    )

    all_claims.setdefault(
        qid,
        {},
    )[claim_index] = rec["claim"]


# Convert recovered claim dictionaries to ordered lists.
all_claims = {
    qid: [
        claims_by_index[i]
        for i in sorted(
            claims_by_index
        )
    ]
    for qid, claims_by_index
    in all_claims.items()
}


# Then recover the dedicated claim-decomposition checkpoint.
for rec in _load_jsonl(
    CLAIM_DECOMPOSITION_CHECKPOINT_PATH
):

    qid = rec["question_id"]

    claims = rec.get(
        "claims",
        None,
    )

    # Only accept valid completed checkpoint entries.
    if isinstance(
        claims,
        list,
    ):
        all_claims[qid] = claims


done = set(
    all_claims.keys()
)

print(
    f"Resuming: {len(done)} of {len(df)} "
    "answers already have claims."
)


# ---------------------------------------------------------------------
# Decompose remaining answers
# ---------------------------------------------------------------------

for qid, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Claim decomposition (Gemini Flash-Lite)",
):

    if qid in done:
        continue

    claims = decompose_answer_into_claims(
        row["answer_text"]
    )

    # Failed Gemini calls are NOT written to the checkpoint.
    # This allows the notebook to retry them later.
    if claims is None:
        continue

    all_claims[qid] = claims

    _append_jsonl(
        CLAIM_DECOMPOSITION_CHECKPOINT_PATH,
        {
            "question_id": qid,
            "question": str(
                row["question"]
            ),
            "claims": claims,
        },
    )

    # Small pause to reduce rate-limit pressure.
    time.sleep(0.4)


# ---------------------------------------------------------------------
# Hard gate
# ---------------------------------------------------------------------
# Never allow downstream NLI verification to treat an API failure as
# "zero claims".

pending = [
    qid
    for qid in df.index
    if qid not in all_claims
]

if pending:

    raise RuntimeError(
        "\nClaim decomposition is INCOMPLETE.\n"
        f"{len(pending)}/{len(df)} answers are still pending.\n"
        f"Pending question_ids: {pending}\n\n"
        "Successful decompositions were saved to the checkpoint.\n"
        "Re-run Section 4 to resume only the pending answers.\n"
        "Do not continue to NLI verification until this list is empty."
    )


# ---------------------------------------------------------------------
# Decomposition quality diagnostics
# ---------------------------------------------------------------------

claim_frame = pd.DataFrame([
    {
        "question_id": qid,
        "claim_index": claim_index,
        "claim": claim,
    }
    for qid, claims in all_claims.items()
    for claim_index, claim in enumerate(claims)
])


print(
    f"Total claims extracted: "
    f"{len(claim_frame)}"
)


if len(claim_frame) > 0:

    claims_per_answer = (
        claim_frame
        .groupby(
            "question_id"
        )
        .size()
    )

    print(
        "Claims per answer: "
        f"mean={claims_per_answer.mean():.2f}, "
        f"min={claims_per_answer.min()}, "
        f"max={claims_per_answer.max()}"
    )

    empty_claims = int(
        (
            claim_frame["claim"]
            .str.strip()
            == ""
        ).sum()
    )

    duplicate_claims = int(
        claim_frame
        .groupby(
            "question_id"
        )["claim"]
        .apply(
            lambda values:
            values.duplicated().sum()
        )
        .sum()
    )

else:

    empty_claims = 0
    duplicate_claims = 0


print(
    f"Empty claims: "
    f"{empty_claims}"
)

print(
    f"Duplicate claims within an answer: "
    f"{duplicate_claims}"
)


print(
    "\nExample claims (first answer):"
)

first_qid = df.index[0]

for claim in all_claims.get(
    first_qid,
    [],
)[:6]:

    print(
        f"  - {claim}"
    )

Resuming: 100 of 100 answers already have claims.


Claim decomposition (Gemini Flash-Lite): 100%|██████████| 100/100 [00:00<00:00, 8599.29it/s]

Total claims extracted: 644
Claims per answer: mean=6.44, min=2, max=17
Empty claims: 0
Duplicate claims within an answer: 0

Example claims (first answer):
  - 36.4% of Pap smears with glandular cell abnormalities were flagged as "review for abnormality" on rapid prescreening.
  - In 54.2% of patients who had histologic follow-up after a Pap smear with glandular cell abnormalities, the Pap smears that triggered their biopsy had been flagged as "review for abnormality" on rapid prescreening.


---
# 5. Hallucination Detection (BEFORE control)

In [5]:
# Section 5a - Load NLI model + biomedical encoder (used by the relevance gate).
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.to(device)
nli_model.eval()
nli_id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
print(f"NLI label mapping: {nli_id2label}")

if USE_BIOMEDICAL_RELEVANCE_GATE:
    biomedical_encoder = SentenceTransformer(EMBEDDING_MODEL_PUBMEDBERT)
    _bio_embed_cache = {}
else:
    biomedical_encoder = None
    _bio_embed_cache = None


def _nli_batch(premises, hypotheses):
    """Run NLI on a batch of (premise, hypothesis) pairs -> [(label, score, probs), ...]."""
    if len(premises) == 0:
        return []
    inputs = nli_tokenizer(
        list(premises),
        list(hypotheses),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512,
    ).to(device)
    with torch.no_grad():
        probs = torch.softmax(nli_model(**inputs).logits, dim=-1).cpu().numpy()
    out = []
    for p in probs:
        label_probs = {nli_id2label[i]: float(p[i]) for i in range(len(p))}
        label = max(label_probs, key=label_probs.get)
        out.append((label, label_probs[label], label_probs))
    return out


def _bio_similarity(claim, evidence_text):
    """Cosine similarity of claim and evidence under the biomedical encoder (cached)."""
    if biomedical_encoder is None:
        return None
    for key, text in ((f"c:{claim}", claim), (f"e:{evidence_text}", evidence_text)):
        if key not in _bio_embed_cache:
            _bio_embed_cache[key] = biomedical_encoder.encode(
                [text], normalize_embeddings=True, convert_to_numpy=True
            )[0]
    v1, v2 = _bio_embed_cache[f"c:{claim}"], _bio_embed_cache[f"e:{evidence_text}"]
    return float(np.dot(v1, v2))


def verify_claim(claim, chunks, need_bio_sim=True):
    """NLI-verify one claim against a list of evidence chunks -> list of per-chunk results."""
    results = []
    keep = [c for c in chunks if (c.get("chunk_text") or "").strip()]
    if not keep:
        return []
    premises = [c.get("chunk_text").strip() for c in keep]
    hypotheses = [claim] * len(keep)
    batch = _nli_batch(premises, hypotheses)
    for chunk, (label, score, probs) in zip(keep, batch):
        evidence_text = chunk.get("chunk_text").strip()
        results.append({
            "chunk_id": chunk.get("chunk_id"),
            "pmid": chunk.get("pmid"),
            "nli_label": label,
            "nli_score": score,
            "label_probs": probs,
            "bio_similarity": _bio_similarity(claim, evidence_text) if need_bio_sim else None,
            "evidence_text": evidence_text,
        })
    return results


def decide_verification(results, entail_thr, contrad_thr, use_bio_gate=True):
    """Aggregate per-chunk NLI results into one verification label.

    Strongest entailment >= entail_thr and (optionally) passing the biomedical gate
    -> SUPPORTED; else strongest contradiction >= contrad_thr -> CONTRADICTED; else
    NOT_ENOUGH_EVIDENCE. Returns (verification_label, best_result_or_None).
    """
    if not results:
        return "NOT_ENOUGH_EVIDENCE", None

    ents = [
        r for r in results
        if r["nli_label"] == "entailment" and r["nli_score"] >= entail_thr
    ]
    if use_bio_gate and USE_BIOMEDICAL_RELEVANCE_GATE:
        ents = [
            r for r in ents
            if r["bio_similarity"] is not None and r["bio_similarity"] >= MIN_CLAIM_EVIDENCE_SIMILARITY
        ]
    if ents:
        best = max(ents, key=lambda r: r["nli_score"])
        return "SUPPORTED", best

    cons = [
        r for r in results
        if r["nli_label"] == "contradiction" and r["nli_score"] >= contrad_thr
    ]
    if cons:
        best = max(cons, key=lambda r: r["nli_score"])
        return "CONTRADICTED", best

    best = max(results, key=lambda r: r["nli_score"])
    return "NOT_ENOUGH_EVIDENCE", best


Device: cpu


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI label mapping: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
# Section 5b - Verify every claim (resumable via claim_verification.jsonl).
existing_records = {rec["claim_id"]: rec for rec in _load_jsonl(CLAIM_VERIFICATION_JSONL_PATH)}

claim_records = []
for qid, row in tqdm(df.iterrows(), total=len(df), desc="Verifying claims (NLI)"):
    claims = all_claims.get(qid, [])
    kept_chunks = row["kept_chunks_parsed"]

    for claim_index, claim in enumerate(claims):
        claim_id = f"{qid}_{claim_index}"
        old = existing_records.get(claim_id)
        if old is not None and old.get("claim") == claim:
            claim_records.append(old)
            continue

        chunk_results = verify_claim(claim, kept_chunks, need_bio_sim=USE_BIOMEDICAL_RELEVANCE_GATE)
        verification_label, best = decide_verification(
            chunk_results, NLI_ENTAILMENT_THRESHOLD, NLI_CONTRADICTION_THRESHOLD
        )

        record = {
            "question_id": qid,
            "claim_id": claim_id,
            "claim": claim,
            "verification_label": verification_label,
            "best_evidence_chunk_id": best["chunk_id"] if best else None,
            "best_evidence_text": best["evidence_text"] if best else None,
            "nli_score": best["nli_score"] if best else None,
            "bio_similarity": best["bio_similarity"] if best else None,
            "action": VERIFICATION_LABEL_TO_ACTION[verification_label],
            "reason": {
                "SUPPORTED": "Evidence supports the claim.",
                "CONTRADICTED": "Evidence contradicts the claim.",
                "NOT_ENOUGH_EVIDENCE": "Evidence does not support or contradict the claim (coverage gap).",
            }[verification_label],
        }
        claim_records.append(record)
        _append_jsonl(CLAIM_VERIFICATION_JSONL_PATH, record)

claims_df = (
    pd.DataFrame(claim_records)
    .drop_duplicates(subset="claim_id", keep="last")
    .sort_values(["question_id", "claim_id"])
    .reset_index(drop=True)
)

# Hard gate: every one of the 100 questions must have verified claims.
q_with_claims = set(claims_df["question_id"])
missing_q = [qid for qid in df.index if qid not in q_with_claims]
if missing_q:
    raise RuntimeError(
        "Detection is INCOMPLETE: " + str(len(missing_q)) +
        " answers have no verified claims: " + str(missing_q) +
        ". Re-run Section 4 to finish claim decomposition first."
    )

print(f"Total claim-level records: {len(claims_df)}")
print("\nVerification label counts:")
print(claims_df["verification_label"].value_counts().to_string())

if claims_df["bio_similarity"].notna().sum() > 0:
    print("\nBiomedical similarity (claim vs best evidence) distribution:")
    print(claims_df["bio_similarity"].describe().to_string())


# ---- Answer-level summary (hallucination risk) ----
answer_records = []
for qid, row in df.iterrows():
    question_claims = claims_df[claims_df["question_id"] == qid]
    total = len(question_claims)
    supported = int((question_claims["verification_label"] == "SUPPORTED").sum())
    contradicted = int((question_claims["verification_label"] == "CONTRADICTED").sum())
    neutral = int((question_claims["verification_label"] == "NOT_ENOUGH_EVIDENCE").sum())

    if total == 0:
        support_rate = contradiction_rate = missing_evidence_rate = hallucination_risk = 0.0
    else:
        support_rate = supported / total
        contradiction_rate = contradicted / total
        missing_evidence_rate = neutral / total
        hallucination_risk = (contradicted + neutral) / total

    answer_records.append({
        "question_id": qid,
        "question": row["question"],
        "generated_answer": row["answer_text"],
        "total_claims": total,
        "supported_claims": supported,
        "contradicted_claims": contradicted,
        "neutral_claims": neutral,
        "support_rate": support_rate,
        "contradiction_rate": contradiction_rate,
        "missing_evidence_rate": missing_evidence_rate,
        "hallucination_risk": hallucination_risk,
    })

answer_summary_df = pd.DataFrame(answer_records)
print("\nAnswer-level hallucination summary (first 10):")
print(answer_summary_df[["question_id", "total_claims", "support_rate",
                        "contradiction_rate", "missing_evidence_rate", "hallucination_risk"]]
      .head(10).to_string())


# ---- NLI false-positive / false-negative diagnostics ----
def _numbers(text):
    return re.findall(r"\d+(?:\.\d+)?", text or "")


def _tokens(text):
    return set(re.findall(r"[a-z0-9]+", (text or "").lower()))


diagnostics = []
for _, r in claims_df.iterrows():
    claim_numbers = _numbers(r["claim"])
    evidence = r["best_evidence_text"] or ""

    if r["verification_label"] == "SUPPORTED" and claim_numbers and evidence:
        missing = [n for n in claim_numbers if n not in _numbers(evidence)]
        if missing:
            diagnostics.append({
                "claim_id": r["claim_id"],
                "question_id": r["question_id"],
                "claim": r["claim"],
                "verification_label": r["verification_label"],
                "issue": "FP? numerical mismatch",
                "detail": f"numbers absent from evidence: {missing}",
            })
    elif r["verification_label"] == "NOT_ENOUGH_EVIDENCE" and evidence:
        overlap = len(_tokens(r["claim"]) & _tokens(evidence)) / max(1, len(_tokens(r["claim"])))
        if overlap >= 0.5:
            diagnostics.append({
                "claim_id": r["claim_id"],
                "question_id": r["question_id"],
                "claim": r["claim"],
                "verification_label": r["verification_label"],
                "issue": "FN? high lexical overlap",
                "detail": f"token overlap {overlap:.2f}",
            })

nli_diagnostics_df = pd.DataFrame(diagnostics)


# ---- Save detection outputs ----
claims_csv_df = claims_df.drop(columns=["best_evidence_text"], errors="ignore").copy()
claims_csv_df.to_csv(CLAIM_VERIFICATION_CSV_PATH, index=False)
answer_summary_df.to_csv(ANSWER_SUMMARY_CSV_PATH, index=False)
with open(ANSWER_SUMMARY_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, r in answer_summary_df.iterrows():
        f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")
nli_diagnostics_df.to_csv(NLI_DIAGNOSTICS_CSV_PATH, index=False)

print(f"\nSaved: {CLAIM_VERIFICATION_CSV_PATH}")
print(f"Saved: {ANSWER_SUMMARY_CSV_PATH}")
print(f"Saved: {NLI_DIAGNOSTICS_CSV_PATH} (diagnostics: {len(nli_diagnostics_df)})")

# ---- DETECTION SUMMARY (BEFORE the hallucination control) ----
print("\n========== HALLUCINATION DETECTION SUMMARY (BEFORE CONTROL) ==========")
print(f"Answers processed          : {len(answer_summary_df)} / {REQUIRED_N}")
print(f"Total claims               : {len(claims_df)}")
n_sup = int((claims_df["verification_label"] == "SUPPORTED").sum())
n_con = int((claims_df["verification_label"] == "CONTRADICTED").sum())
n_ne = int((claims_df["verification_label"] == "NOT_ENOUGH_EVIDENCE").sum())
print(f"  SUPPORTED           : {n_sup} ({n_sup / len(claims_df):.2%})")
print(f"  CONTRADICTED        : {n_con} ({n_con / len(claims_df):.2%})")
print(f"  NOT_ENOUGH_EVIDENCE : {n_ne} ({n_ne / len(claims_df):.2%})")
print(f"Unsupported rate (contradicted + missing evidence): {(n_con + n_ne) / len(claims_df):.2%}")
print(f"Mean hallucination risk (per answer): {answer_summary_df['hallucination_risk'].mean():.3f}")


Verifying claims (NLI): 100%|██████████| 100/100 [00:00<00:00, 593.97it/s]


Total claim-level records: 644

Verification label counts:
verification_label
SUPPORTED              465
NOT_ENOUGH_EVIDENCE    156
CONTRADICTED            23

Biomedical similarity (claim vs best evidence) distribution:
count    644.000000
mean       0.935424
std        0.027572
min        0.840494
25%        0.918056
50%        0.937717
75%        0.955917
max        0.986707

Answer-level hallucination summary (first 10):
   question_id  total_claims  support_rate  contradiction_rate  missing_evidence_rate  hallucination_risk
0            0             2      0.500000            0.000000               0.500000            0.500000
1            1             8      1.000000            0.000000               0.000000            0.000000
2            2             4      1.000000            0.000000               0.000000            0.000000
3            3             6      0.166667            0.000000               0.833333            0.833333
4            4            10      1.00000

---
# 6. Hallucination Mitigation (the control module)

In [7]:
# ---------------------------------------------------------------------
# Retrieval stack for mitigation re-retrieval
# ---------------------------------------------------------------------
# IMPORTANT:
# Use qdrant_client instead of client.
#
# "client" may already be used by the Gemini SDK in this notebook.
# Keeping separate names prevents Gemini and Qdrant from overwriting
# each other in the Jupyter kernel.
# ---------------------------------------------------------------------

chunk_df = pd.read_parquet(
    PROCESSED_DIR / "pubmedqa_chunks.parquet"
)

chunk_minilm = np.load(
    PROCESSED_DIR / "pubmedqa_chunk_embeddings.npy"
)

chunk_pubmed = np.load(
    PROCESSED_DIR / "pubmedqa_chunk_embeddings_pubmedbert.npy"
)


# Validate that every chunk has an embedding in both indexes.
if not (
    len(chunk_df)
    == len(chunk_minilm)
    == len(chunk_pubmed)
):
    raise ValueError(
        "Chunk count and embedding counts do not match."
    )


print(
    f"Chunks: {len(chunk_df)} | "
    f"MiniLM embeddings: {chunk_minilm.shape} | "
    f"PubMedBERT embeddings: {chunk_pubmed.shape}"
)


# ---------------------------------------------------------------------
# Load the two embedding models used for mitigation re-retrieval.
# ---------------------------------------------------------------------

embedding_minilm = SentenceTransformer(
    EMBEDDING_MODEL_MINILM
)

embedding_pubmed = SentenceTransformer(
    EMBEDDING_MODEL_PUBMEDBERT
)


# Cross-encoder used after dense fusion.
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length=512,
)


# ---------------------------------------------------------------------
# Create a dedicated Qdrant client.
# ---------------------------------------------------------------------
# Do NOT call this variable "client", because Gemini also uses a client.

qdrant_client = QdrantClient(
    ":memory:"
)


# ---------------------------------------------------------------------
# Build the two Qdrant collections.
# ---------------------------------------------------------------------

for collection_name, vectors in (
    (
        COLLECTION_MINILM,
        chunk_minilm,
    ),
    (
        COLLECTION_PUBMEDBERT,
        chunk_pubmed,
    ),
):

    # Recreate each collection from scratch.
    qdrant_client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vectors.shape[1],
            distance=Distance.COSINE,
        ),
    )

    # Store the PubMedQA metadata with each vector.
    payloads = chunk_df.to_dict(
        orient="records"
    )

    points = [
        PointStruct(
            id=i,
            vector=vectors[i].tolist(),
            payload=payloads[i],
        )
        for i in range(len(vectors))
    ]

    qdrant_client.upsert(
        collection_name=collection_name,
        points=points,
    )

    points_count = (
        qdrant_client
        .get_collection(collection_name)
        .points_count
    )

    print(
        f"Collection '{collection_name}': "
        f"{points_count} points"
    )


# ---------------------------------------------------------------------
# Hybrid re-retrieval
# ---------------------------------------------------------------------

def hybrid_reretrieve(
    queries,
    exclude_chunk_ids=None,
    top_k=RERETRIEVE_KEEP_TOP_N,
):
    """
    Retrieve additional evidence for an unresolved claim.

    Pipeline:
    1. Search both dense indexes:
       - MiniLM
       - PubMedBERT
    2. Fuse rankings using Reciprocal Rank Fusion (RRF).
    3. Re-rank fused candidates with the cross-encoder.
    4. Remove low-relevance evidence.
    5. Return the strongest new evidence chunks.

    Parameters
    ----------
    queries : str or list[str]
        One or several query formulations.

    exclude_chunk_ids : set/list
        Evidence chunks already used by the original RAG retrieval.

    top_k : int
        Maximum number of final evidence chunks to keep.
    """

    exclude_chunk_ids = set(
        exclude_chunk_ids or []
    )


    # Make sure the function accepts either:
    #     "query"
    # or:
    #     ["query 1", "query 2"]
    if isinstance(
        queries,
        str,
    ):
        queries = [queries]


    candidates = {}


    # -----------------------------------------------------------------
    # Run both query formulations over both dense indexes.
    # -----------------------------------------------------------------

    for query in queries:

        retrieval_indexes = (
            (
                embedding_minilm,
                COLLECTION_MINILM,
            ),
            (
                embedding_pubmed,
                COLLECTION_PUBMEDBERT,
            ),
        )


        for embedding_model, collection_name in retrieval_indexes:

            # Encode the retrieval query.
            query_vector = embedding_model.encode(
                [query],
                normalize_embeddings=True,
                convert_to_numpy=True,
            )[0]


            # Search Qdrant.
            hits = qdrant_client.query_points(
                collection_name=collection_name,
                query=query_vector.tolist(),
                limit=RERETRIEVE_TOP_K_PER_INDEX,
            ).points


            # Add each retrieved chunk to the fusion pool.
            for rank, hit in enumerate(
                hits,
                start=1,
            ):

                payload = (
                    hit.payload
                    or {}
                )

                chunk_id = payload.get(
                    "chunk_id"
                )


                # Do not re-use chunks that the original RAG stage already used.
                if chunk_id in exclude_chunk_ids:
                    continue


                candidate = candidates.setdefault(
                    hit.id,
                    {
                        "chunk_id": chunk_id,
                        "pmid": payload.get(
                            "pmid"
                        ),
                        "chunk_text": payload.get(
                            "chunk_text"
                        ),
                        "rrf": 0.0,
                    },
                )


                # Reciprocal Rank Fusion.
                candidate["rrf"] += (
                    1.0
                    / (
                        RRF_K
                        + rank
                    )
                )


    # -----------------------------------------------------------------
    # Keep strongest fused candidates.
    # -----------------------------------------------------------------

    fused = sorted(
        candidates.values(),
        key=lambda c: c["rrf"],
        reverse=True,
    )[
        :RERETRIEVE_TOP_K_PER_INDEX
    ]


    if not fused:
        return []


    # -----------------------------------------------------------------
    # Cross-encoder relevance re-ranking.
    # -----------------------------------------------------------------
    #
    # Because two query variants may have been used, score each candidate
    # against all query variants and keep its best score.
    # -----------------------------------------------------------------

    for candidate in fused:

        pair_scores = reranker.predict(
            [
                (
                    query,
                    candidate["chunk_text"],
                )
                for query in queries
            ],
            show_progress_bar=False,
        )

        candidate["relevance"] = float(
            np.max(
                pair_scores
            )
        )


    # -----------------------------------------------------------------
    # Apply relevance threshold.
    # -----------------------------------------------------------------

    kept = [
        candidate
        for candidate in fused
        if candidate["relevance"]
        >= MIN_RERETRIEVAL_RELEVANCE
    ]


    # Highest cross-encoder relevance first.
    kept = sorted(
        kept,
        key=lambda c: c["relevance"],
        reverse=True,
    )


    return kept[:top_k]


print(
    "Mitigation retrieval stack ready."
)

print(
    f"Qdrant client type: "
    f"{type(qdrant_client).__name__}"
)

Chunks: 1483 | MiniLM embeddings: (1483, 384) | PubMedBERT embeddings: (1483, 768)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

C:\Users\YOGA\AppData\Local\Temp\ipykernel_13180\1199644516.py:89: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


Collection 'pubmedqa_chunks': 1483 points


C:\Users\YOGA\AppData\Local\Temp\ipykernel_13180\1199644516.py:89: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


Collection 'pubmedqa_chunks_pubmedbert': 1483 points
Mitigation retrieval stack ready.
Qdrant client type: QdrantClient


In [8]:
# Section 6b - Gemini helpers for mitigation (query generation + correction).
def _gemini_text(prompt):
    """Call Gemini with the robust retry/backoff and return raw text (or None)."""
    response = call_gemini_robust(prompt)
    return response.text.strip() if response is not None else None


def generate_retrieval_query_for_claim(claim, evidence=None):
    """Generate a targeted retrieval query from a claim's key entities."""
    evidence = evidence or "No evidence text available."
    prompt = (
        "You are a medical evidence retrieval assistant.\n\n"
        "The following medical claim could not be verified because the available evidence was insufficient.\n\n"
        "Claim:\n" + claim + "\n\n"
        "Current evidence:\n" + evidence + "\n\n"
        "Task:\n"
        "Generate ONE targeted retrieval query to find the missing medical evidence needed to verify this claim.\n\n"
        "Instructions:\n"
        "- Focus on the key medical entities: disease, drug, dose, treatment, biomarker, population, and the relationship stated in the claim.\n"
        "- Preserve numbers, percentages and negations that are essential to the claim.\n"
        "- Do not answer the claim and do not assume it is true or false.\n"
        "- Return only the query text, no preamble.\n\n"
        "Retrieval query:"
    )
    response_text = _gemini_text(prompt)
    if response_text is None:
        return claim  # fallback: use the claim itself as the query
    query = response_text.strip().splitlines()[0].strip()
    return query if query else claim


def _parse_correction_json(text):
    """Parse the JSON object returned by Gemini for correction/grounding."""
    if not text or not text.strip():
        return None
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"\s*```\s*$", "", text).strip()
    try:
        obj = json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None
        try:
            obj = json.loads(match.group(0))
        except Exception:
            return None
    return obj if isinstance(obj, dict) else None


def _correction_feedback(attempt, strategy):
    """Retry feedback for the LLM when a previous rewrite was rejected by NLI."""
    if attempt <= 1:
        return ""
    if strategy == "ground":
        return ("\n\nFeedback: the previous grounded claim was rejected because NLI did not confirm it "
                "against the evidence. Restate the evidence's facts almost literally (same numbers and "
                "wording), even if that makes the sentence shorter.")
    return ("\n\nFeedback: the previous corrected claim was rejected because NLI did not confirm it "
            "against the evidence. Make the corrected claim a near-literal restatement of the "
            "evidence's facts (same numbers and wording for the contradicted part).")


def correct_contradicted_claim(claim, evidence, attempt=1):
    """Rewrite a CONTRADICTED claim using only verified evidence.

    The evidence is treated as the ground truth: the corrected claim must restate the
    evidence's facts (replacing the contradicted numbers/relations/negations) and drop
    anything the evidence does not support. Returns
    {"corrected_claim", "correction_reason", "declined": bool, "raw": str}.
    """
    evidence = evidence or "No evidence text available."
    feedback = _correction_feedback(attempt, "correct")
    prompt = (
        "You are a strict medical fact-correction assistant for a RAG pipeline.\n\n"
        "A generated medical claim was flagged as CONTRADICTED: the retrieved evidence states a "
        "different fact. The evidence comes from a trusted biomedical corpus and is the ground "
        "truth for correction.\n\n"
        "Task: rewrite the claim so every contradicted part now matches the evidence, and drop any "
        "part the evidence does not support.\n\n"
        "Rules:\n"
        "1. State ONLY facts that are literally present in the evidence. Never invent or infer "
        "numbers, outcomes or relations.\n"
        "2. When the evidence gives a different number/percentage/relation/negation for the same "
        "entity, use the evidence's exact value and wording.\n"
        "3. Keep the parts of the claim that the evidence still supports.\n"
        "4. The output must be a single standalone factual sentence (or short statement).\n"
        "5. Decline ONLY when the evidence is about a completely different topic (no entity "
        "overlap): set corrected_claim to an empty string and declined to true.\n"
        "6. Return JSON only." + feedback + "\n\n"
        "Original claim:\n" + claim + "\n\n"
        "Evidence:\n" + evidence + "\n\n"
        'Output JSON:\n{"corrected_claim": "...", "correction_reason": "...", "declined": true/false}'
    )
    response_text = _gemini_text(prompt)
    if response_text is None:
        return {"corrected_claim": CORRECTION_FAILED_TEXT, "correction_reason": "Gemini API error.",
                "declined": False, "raw": ""}
    obj = _parse_correction_json(response_text)
    if obj is None:
        return {"corrected_claim": response_text.strip(), "correction_reason": "", "declined": False,
                "raw": response_text}
    corrected = str(obj.get("corrected_claim") or "").strip()
    return {"corrected_claim": corrected,
            "correction_reason": str(obj.get("correction_reason") or ""),
            "declined": bool(obj.get("declined")), "raw": response_text}


def ground_claim_to_evidence(claim, evidence_context, attempt=1):
    """Rewrite a claim as the closest factual statement the retrieved evidence supports.

    Bounded fallback used when bounded re-retrieval cannot fully verify the claim. The
    rewritten claim is still gated by verify-before-accept (see Section 6c). Returns
    {"grounded_claim", "correction_reason", "declined": bool, "raw": str}.
    """
    evidence_context = (evidence_context or "").strip() or "No evidence text available."
    feedback = _correction_feedback(attempt, "ground")
    prompt = (
        "You are a medical claim grounding assistant for a RAG pipeline.\n\n"
        "A generated medical claim could not be verified from the retrieved evidence. Below are the "
        "closest evidence chunks found for the same question. Produce the closest factual statement "
        "about the SAME medical topic that is directly supported by the evidence.\n\n"
        "Rules:\n"
        "1. Base the statement ONLY on facts literally present in the evidence.\n"
        "2. If the evidence states a concrete fact on the claim's topic (number, percentage, "
        "relation, outcome, negation), rewrite the claim to state that fact using the evidence's "
        "own wording.\n"
        "3. If the evidence has NO fact on the claim's topic, decline: set grounded_claim to an "
        "empty string and declined to true.\n"
        "4. The output must be a single standalone factual statement. Do not add explanations.\n"
        "5. Return JSON only." + feedback + "\n\n"
        "Original claim:\n" + claim + "\n\n"
        "Closest evidence:\n" + evidence_context + "\n\n"
        'Output JSON:\n{"grounded_claim": "...", "correction_reason": "...", "declined": true/false}'
    )
    response_text = _gemini_text(prompt)
    if response_text is None:
        return {"grounded_claim": CORRECTION_FAILED_TEXT, "correction_reason": "Gemini API error.",
                "declined": False, "raw": ""}
    obj = _parse_correction_json(response_text)
    if obj is None:
        return {"grounded_claim": response_text.strip(), "correction_reason": "", "declined": False,
                "raw": response_text}
    grounded = str(obj.get("grounded_claim") or "").strip()
    return {"grounded_claim": grounded,
            "correction_reason": str(obj.get("correction_reason") or ""),
            "declined": bool(obj.get("declined")), "raw": response_text}


In [9]:
# Section 6c - Mitigation state machine (checkpointed + resumable).

def _evidence_chunk(text, chunk_id=None, pmid=None):
    return {"chunk_id": chunk_id, "pmid": pmid, "chunk_text": text}


def verify_claim_against_new_evidence(claim, new_chunks):
    """Re-verify a claim against newly retrieved evidence chunks."""
    if not new_chunks:
        return {
            "new_verification_label": "NOT_ENOUGH_EVIDENCE",
            "best_new_evidence_chunk_id": None,
            "best_new_evidence_text": None,
            "new_nli_score": None,
            "all_new_evidence_candidates": [],
        }
    chunk_results = verify_claim(claim, new_chunks, need_bio_sim=USE_BIOMEDICAL_RELEVANCE_GATE)
    verification_label, best = decide_verification(
        chunk_results, NLI_ENTAILMENT_THRESHOLD, NLI_CONTRADICTION_THRESHOLD
    )
    return {
        "new_verification_label": verification_label,
        "best_new_evidence_chunk_id": best["chunk_id"] if best else None,
        "best_new_evidence_text": best["evidence_text"] if best else None,
        "new_nli_score": best["nli_score"] if best else None,
        "all_new_evidence_candidates": chunk_results,
    }


def _claim_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text or ""))


def _claim_tokens(text):
    return set(re.findall(r"[a-z0-9]+", (text or "").lower()))


def _claim_overlap(claim, evidence):
    a = _claim_tokens(claim)
    if not a:
        return 0.0
    return len(a & _claim_tokens(evidence)) / len(a)


def accept_corrected_claim(corrected_claim, evidence_chunks):
    """Verify-before-accept for an LLM rewrite (evidence-derived rewrite policy).

    Returns (accepted: bool, best_result: dict|None). A rewrite is accepted only when
    at least one evidence chunk satisfies ALL of:
      - strict NLI SUPPORTED, OR (evidence-derived policy) entailment >=
        CORRECTION_ACCEPT_ENTAILMENT and contradiction probability <
        CORRECTION_ACCEPT_MAX_CONTRADICTION;
      - the biomedical relevance gate;
      - numeric fidelity: every number in the rewrite appears in that chunk;
      - a minimum lexical overlap between the rewrite and that chunk.
    """
    keep = [c for c in evidence_chunks if (c.get("chunk_text") or "").strip()]
    if not keep or not (corrected_claim or "").strip():
        return False, None
    results = verify_claim(corrected_claim, keep, need_bio_sim=USE_BIOMEDICAL_RELEVANCE_GATE)

    # Strict path first (identical thresholds as claim detection).
    label, best = decide_verification(results, NLI_ENTAILMENT_THRESHOLD, NLI_CONTRADICTION_THRESHOLD)
    if label == "SUPPORTED" and best is not None:
        return True, best

    # Evidence-derived rewrite policy (relaxed entailment, guarded by contradiction /
    # biomedical similarity / numeric fidelity / lexical overlap).
    cn = _claim_numbers(corrected_claim)
    candidates = []
    for r in results:
        if r["nli_label"] != "entailment":
            continue
        if r["nli_score"] < CORRECTION_ACCEPT_ENTAILMENT:
            continue
        if r["label_probs"].get("contradiction", 0.0) >= CORRECTION_ACCEPT_MAX_CONTRADICTION:
            continue
        if USE_BIOMEDICAL_RELEVANCE_GATE and (r["bio_similarity"] is None
                                              or r["bio_similarity"] < MIN_CLAIM_EVIDENCE_SIMILARITY):
            continue
        ev = r["evidence_text"] or ""
        if cn and not cn.issubset(_claim_numbers(ev)):
            continue
        if _claim_overlap(corrected_claim, ev) < MIN_CORRECTION_LEXICAL_OVERLAP:
            continue
        candidates.append(r)
    if candidates:
        best = max(candidates, key=lambda r: r["nli_score"])
        return True, best
    return False, best


def _question_evidence_pool(question_id, extra_chunks=None):
    """Evidence pool for a question = kept chunks + re-retrieved chunks."""
    pool = [dict(c) for c in df.loc[question_id, "kept_chunks_parsed"]]
    seen = {c.get("chunk_id") for c in pool}
    for chunk in (extra_chunks or []):
        cid = chunk.get("chunk_id")
        if cid in seen:
            continue
        seen.add(cid)
        pool.append(dict(chunk))
    return pool


def _build_grounding_context(attempts, extra_chunks=None, cap=GROUNDING_CONTEXT_CHAR_CAP):
    """Concatenate the most relevant re-retrieved chunks for the grounding prompt."""
    parts = []
    seen = set()
    for attempt in attempts:
        chunks = attempt.get("new_chunks") or []
        best = max(chunks, key=lambda c: c.get("score") or -999.0) if chunks else None
        if best is None:
            continue
        cid = best.get("chunk_id")
        if cid in seen:
            continue
        seen.add(cid)
        parts.append(str(best.get("chunk_text") or "").strip())
    for chunk in (extra_chunks or []):
        cid = chunk.get("chunk_id")
        if cid in seen or not (chunk.get("chunk_text") or "").strip():
            continue
        seen.add(cid)
        parts.append(str(chunk["chunk_text"]).strip())
    text = "\n\n".join(p for p in parts if p)
    if cap and len(text) > cap:
        text = text[:cap]
    return text


def mitigate_claim(row):
    """Apply the hallucination-control decision state machine to ONE claim.

    SUPPORTED -> keep. CONTRADICTED -> evidence-authoritative rewrite (verify-before-
    accept with the evidence-derived rewrite policy). NOT_ENOUGH_EVIDENCE -> bounded
    hybrid re-retrieval loop; if still unresolved, a bounded grounding rewrite; only then
    escalate to human review (UNRESOLVED_HUMAN_REVIEW, removed from the emitted answer).
    """
    question_id = row["question_id"]
    claim_id = row["claim_id"]
    original_claim = row["claim"]
    label = row["verification_label"]
    best_evidence_text = row["best_evidence_text"]
    best_evidence_chunk_id = row["best_evidence_chunk_id"]

    trace = {
        "question_id": question_id,
        "claim_id": claim_id,
        "original_claim": original_claim,
        "verification_label": label,
        "final_evidence_chunk_id": best_evidence_chunk_id,
        "final_evidence_text": best_evidence_text,
        "attempts": [],
        "reason": "",
    }

    def escalate(reason):
        trace.update({
            "final_action": "human_review",
            "final_claim": original_claim,
            "final_status": "UNRESOLVED_HUMAN_REVIEW",
            "reason": reason,
        })
        return trace

    # ---- Case A: SUPPORTED -> keep ------------------------------------
    if label == "SUPPORTED":
        trace.update({
            "final_action": "keep",
            "final_claim": original_claim,
            "final_status": "RESOLVED_SUPPORTED",
            "reason": "Evidence supports the claim.",
        })
        return trace

    # ---- Case B: CONTRADICTED -> correct, then verify-before-accept ----
    if label == "CONTRADICTED":
        pool = _question_evidence_pool(
            question_id,
            [_evidence_chunk(best_evidence_text, best_evidence_chunk_id)],
        )
        for c_attempt in range(1, MAX_CORRECTION_ATTEMPTS + 1):
            correction = correct_contradicted_claim(
                original_claim, best_evidence_text, attempt=c_attempt
            )
            corrected_claim = correction["corrected_claim"]
            if (correction.get("declined")
                    or correction["correction_reason"] == "Gemini API error."
                    or not corrected_claim
                    or corrected_claim.strip().lower() == original_claim.strip().lower()):
                break
            accepted, best = accept_corrected_claim(corrected_claim, pool)
            if accepted:
                trace.update({
                    "final_action": "correct",
                    "final_claim": corrected_claim,
                    "final_status": "RESOLVED_CORRECTED",
                    "final_evidence_chunk_id": best["chunk_id"] if best else None,
                    "final_evidence_text": best["evidence_text"] if best else None,
                    "reason": correction["correction_reason"] or
                              "Claim contradicted evidence and was rewritten using verified evidence (NLI-confirmed).",
                })
                return trace
        return escalate(
            "No rewrite of the contradicted claim could be confirmed by NLI against the evidence "
            "(declined or rejected by the evidence-derived rewrite policy)."
        )

    # ---- Case C: NOT_ENOUGH_EVIDENCE -> bounded re-retrieval loop ------
    if label == "NOT_ENOUGH_EVIDENCE":
        current_evidence_text = best_evidence_text
        excluded = KEPT_CHUNK_IDS_BY_QUESTION.get(question_id, set())
        pool = _question_evidence_pool(question_id, [])
        pool_ids = {c.get("chunk_id") for c in pool}

        for attempt_number in range(1, MAX_MITIGATION_ATTEMPTS + 1):
            query = generate_retrieval_query_for_claim(original_claim, current_evidence_text)
            # Two query variants: the Gemini query and the raw claim itself.
            new_chunks = hybrid_reretrieve([query, original_claim], exclude_chunk_ids=excluded)
            verification = verify_claim_against_new_evidence(original_claim, new_chunks)

            trace["attempts"].append({
                "attempt_number": attempt_number,
                "query": query,
                "queries": [query, original_claim],
                "new_chunks": [{
                    "chunk_id": c["chunk_id"],
                    "pmid": c.get("pmid"),
                    "score": c.get("relevance"),
                    "rrf": c.get("rrf"),
                    "chunk_text": c["chunk_text"],
                } for c in new_chunks],
                "new_verification_label": verification["new_verification_label"],
                "new_nli_score": verification["new_nli_score"],
            })

            # Accumulate the new chunks in the question's evidence pool (for rewrites).
            for c in new_chunks:
                cid = c.get("chunk_id")
                if cid not in pool_ids:
                    pool_ids.add(cid)
                    pool.append(dict(c))

            if verification["new_verification_label"] == "SUPPORTED":
                trace.update({
                    "final_action": "keep_after_reretrieval",
                    "final_claim": original_claim,
                    "final_status": "RESOLVED_AFTER_RERETRIEVAL",
                    "final_evidence_chunk_id": verification["best_new_evidence_chunk_id"],
                    "final_evidence_text": verification["best_new_evidence_text"],
                    "reason": f"New evidence found on attempt {attempt_number} supports the claim.",
                })
                return trace

            if verification["new_verification_label"] == "CONTRADICTED":
                for c_attempt in range(1, MAX_CORRECTION_ATTEMPTS + 1):
                    correction = correct_contradicted_claim(
                        original_claim, verification["best_new_evidence_text"], attempt=c_attempt
                    )
                    corrected_claim = correction["corrected_claim"]
                    if (correction.get("declined")
                            or correction["correction_reason"] == "Gemini API error."
                            or not corrected_claim
                            or corrected_claim.strip().lower() == original_claim.strip().lower()):
                        break
                    accepted, best = accept_corrected_claim(corrected_claim, pool)
                    if accepted:
                        trace.update({
                            "final_action": "correct_after_reretrieval",
                            "final_claim": corrected_claim,
                            "final_status": "RESOLVED_CORRECTED_AFTER_RERETRIEVAL",
                            "final_evidence_chunk_id": best["chunk_id"] if best else None,
                            "final_evidence_text": best["evidence_text"] if best else None,
                            "reason": correction["correction_reason"] or
                                      f"New evidence found on attempt {attempt_number} contradicts the claim; "
                                      "claim was rewritten and NLI-confirmed.",
                        })
                        return trace
                return escalate(
                    f"New evidence on attempt {attempt_number} contradicted the claim and the rewrite "
                    "could not be NLI-confirmed."
                )

            # Still NOT_ENOUGH_EVIDENCE: refine the query with the newest evidence and retry.
            if verification["best_new_evidence_text"]:
                current_evidence_text = verification["best_new_evidence_text"]

        # ---- Grounding fallback after the bounded re-retrieval loop ----
        if USE_GROUNDING_FALLBACK:
            context = _build_grounding_context(trace["attempts"], extra_chunks=pool)
            for g_attempt in range(1, MAX_GROUNDING_ATTEMPTS + 1):
                grounding = ground_claim_to_evidence(original_claim, context, attempt=g_attempt)
                grounded = grounding["grounded_claim"]
                if (grounding.get("declined")
                        or grounding["correction_reason"] == "Gemini API error."
                        or not grounded
                        or grounded.strip().lower() == original_claim.strip().lower()):
                    break
                accepted, best = accept_corrected_claim(grounded, pool)
                if accepted:
                    trace.update({
                        "final_action": "ground",
                        "final_claim": grounded,
                        "final_status": "RESOLVED_GROUNDED",
                        "final_evidence_chunk_id": best["chunk_id"] if best else None,
                        "final_evidence_text": best["evidence_text"] if best else None,
                        "reason": grounding["correction_reason"] or
                                  "Claim rewritten as the closest evidence-supported statement (NLI-confirmed).",
                    })
                    return trace
        # ---- Soft-accept fallback for NEE claims ----
        # If the claim does not strongly contradict any evidence across all attempts
        # and has decent biomedical relevance, keep it instead of escalating.
        if USE_SOFT_ACCEPT_NEE:
            all_new_chunks = []
            for att in trace["attempts"]:
                all_new_chunks.extend(att.get("new_chunks", []))
            all_evidence_texts = (
                [c.get("chunk_text", "") for c in _question_evidence_pool(question_id, all_new_chunks)]
            )
            if all_evidence_texts:
                nli_results = verify_claim(original_claim, [
                    {"chunk_text": t, "chunk_id": None, "pmid": None} for t in all_evidence_texts
                ], need_bio_sim=USE_BIOMEDICAL_RELEVANCE_GATE)
                if nli_results:
                    best_con_prob = max(r["label_probs"].get("contradiction", 0.0) for r in nli_results)
                    best_bio = max((r["bio_similarity"] or 0.0) for r in nli_results)
                    if (best_con_prob < SOFT_ACCEPT_MAX_CONTRADICTION_PROB
                            and best_bio >= SOFT_ACCEPT_MIN_BIO_SIMILARITY):
                        trace.update({
                            "final_action": "keep",
                            "final_claim": original_claim,
                            "final_status": "RESOLVED_SOFT_ACCEPT",
                            "reason": (
                                f"Soft-accepted: no strong contradiction ({best_con_prob:.3f} < "
                                f"{SOFT_ACCEPT_MAX_CONTRADICTION_PROB}) and biomedical "
                                f"relevance = {best_bio:.3f}."
                            ),
                        })
                        return trace
        return escalate(
            f"Insufficient evidence after {MAX_MITIGATION_ATTEMPTS} re-retrieval attempts "
            "(grounding declined or rejected)."
        )

    # ---- Unknown label -> escalate -------------------------------------
    return escalate("Unknown verification label.")


# ---- Run mitigation over all claims (resume from the JSONL checkpoint) ----
trace_records = _load_jsonl(MITIGATION_TRACE_JSONL_PATH)
trace_by_claim = {r["claim_id"]: r for r in trace_records}
records = list(trace_records)
print(f"Resuming mitigation: {len(records)} of {len(claims_df)} claims already processed (skipped).")
if records:
    print("NOTE: a mitigation checkpoint already exists. If you changed the mitigation strategy, delete\n"
          "  data/outputs/hallucination_control/mitigation_trace.jsonl (and the derived files\n"
          "  human_expert_review_log.* and mitigated_rag_answers.*) then re-run this cell to reprocess\n"
          "  every claim with the new logic.")

for _, row in tqdm(claims_df.iterrows(), total=len(claims_df), desc="Applying hallucination mitigation"):
    cid = row["claim_id"]
    if cid in trace_by_claim:
        continue
    trace = mitigate_claim(row)
    records.append(trace)
    _append_jsonl(MITIGATION_TRACE_JSONL_PATH, trace)
    time.sleep(0.3)  # stay within Gemini rate limits

mitigation_trace_df = (
    pd.DataFrame(records)
    .drop_duplicates(subset="claim_id", keep="last")
    .reset_index(drop=True)
)

# Hard gate: every claim must have a mitigation result.
missing_claims = [cid for cid in claims_df["claim_id"] if cid not in set(mitigation_trace_df["claim_id"])]
if missing_claims:
    raise RuntimeError(
        "Mitigation is INCOMPLETE: " + str(len(missing_claims)) +
        " claims were not processed because Gemini kept failing.\n"
        "Missing claim_ids: " + str(missing_claims[:15]) + "\n"
        "Progress was saved to the checkpoint file - re-run THIS cell to resume."
    )

# The human-expert review log is derived from the trace: every escalated claim.
human_review_df = mitigation_trace_df[mitigation_trace_df["final_status"] == "UNRESOLVED_HUMAN_REVIEW"].copy()
human_review_df = human_review_df.rename(columns={"original_claim": "claim"})
human_review_df = human_review_df.drop(
    columns=["attempts", "verification_label", "final_evidence_chunk_id", "final_evidence_text"],
    errors="ignore",
)
human_review_df["status"] = "OPEN"
human_review_df["reviewer_notes"] = ""
human_review_df["reviewed_at"] = ""

print(f"\nMitigation applied to {len(mitigation_trace_df)} claims.")
print("\nFinal status counts:")
print(mitigation_trace_df["final_status"].value_counts().to_string())

n_corrected = int(mitigation_trace_df["final_action"].isin(
    ["correct", "correct_after_reretrieval", "ground"]).sum())
n_reretrieved = int((mitigation_trace_df["final_action"] == "keep_after_reretrieval").sum())
print(f"\nAutomatic recoveries: {n_corrected} corrected/grounded (rewritten and NLI-confirmed) + "
      f"{n_reretrieved} recovered via re-retrieval.")
print(f"Claims escalated to human-expert review: {len(human_review_df)}")


Resuming mitigation: 644 of 644 claims already processed (skipped).
NOTE: a mitigation checkpoint already exists. If you changed the mitigation strategy, delete
  data/outputs/hallucination_control/mitigation_trace.jsonl (and the derived files
  human_expert_review_log.* and mitigated_rag_answers.*) then re-run this cell to reprocess
  every claim with the new logic.


Applying hallucination mitigation: 100%|██████████| 644/644 [00:00<00:00, 32557.88it/s]


Mitigation applied to 644 claims.

Final status counts:
final_status
RESOLVED_SUPPORTED                      465
RESOLVED_GROUNDED                        83
UNRESOLVED_HUMAN_REVIEW                  69
RESOLVED_CORRECTED                       13
RESOLVED_AFTER_RERETRIEVAL                9
RESOLVED_CORRECTED_AFTER_RERETRIEVAL      5

Automatic recoveries: 101 corrected/grounded (rewritten and NLI-confirmed) + 9 recovered via re-retrieval.
Claims escalated to human-expert review: 69


---
# 7. Reconstruct Final Controlled Answers

In [10]:
# Section 7 - Reconstruct mitigated answers from the resolved claims.
mitigated_answer_records = []
for qid in mitigation_trace_df["question_id"].unique():
    question_claims = mitigation_trace_df[mitigation_trace_df["question_id"] == qid].copy()
    question_claims["claim_index"] = question_claims["claim_id"].apply(
        lambda cid: int(str(cid).rsplit("_", 1)[-1]) if str(cid).rsplit("_", 1)[-1].isdigit() else 0
    )
    question_claims = question_claims.sort_values("claim_index")

    answer_row = answer_summary_df[answer_summary_df["question_id"] == qid]
    original_answer = answer_row.iloc[0]["generated_answer"] if len(answer_row) else ""
    question_text = answer_row.iloc[0]["question"] if len(answer_row) else ""
    gold_answer = df.loc[qid, "gold_long_answer"] if qid in df.index else ""
    gold_decision = df.loc[qid, "gold_final_decision"] if qid in df.index else ""
    gold_pmid = df.loc[qid, "gold_pmid"] if qid in df.index else ""

    # Emit only resolved claims (escalated claims are removed pending expert review).
    answer_sentences = [
        c["final_claim"]
        for _, c in question_claims.iterrows()
        if c["final_status"] != "UNRESOLVED_HUMAN_REVIEW"
    ]
    mitigated_answer = " ".join(answer_sentences).strip()

    total_claims = len(question_claims)
    resolved = int(question_claims["final_status"].str.startswith("RESOLVED").sum())
    human_review_claims = int((question_claims["final_status"] == "UNRESOLVED_HUMAN_REVIEW").sum())
    reretrieved = int(question_claims["attempts"].apply(len).gt(0).sum())
    corrected = int(question_claims["final_action"].isin(
        ["correct", "correct_after_reretrieval", "ground"]).sum())

    mitigated_answer_records.append({
        "question_id": qid,
        "question": question_text,
        "original_answer": original_answer,
        "mitigated_answer": mitigated_answer,
        "gold_long_answer": gold_answer,
        "gold_final_decision": gold_decision,
        "gold_pmid": gold_pmid,
        "total_claims": int(total_claims),
        "resolved_claims": resolved,
        "corrected_claims": corrected,
        "reretrieved_claims": reretrieved,
        "human_review_claims": human_review_claims,
        "needs_human_review": bool(human_review_claims > 0),
        "mitigation_success_rate": resolved / total_claims if total_claims else 0.0,
        "unresolved_rate": human_review_claims / total_claims if total_claims else 0.0,
    })

mitigated_answers_df = (
    pd.DataFrame(mitigated_answer_records)
    .sort_values("question_id")
    .reset_index(drop=True)
)

# Hard gate: the controlled condition must also cover all 100 questions.
missing_q = [qid for qid in df.index if qid not in set(mitigated_answers_df["question_id"])]
if missing_q:
    raise RuntimeError(
        "Controlled answers are INCOMPLETE: " + str(len(missing_q)) +
        " questions are missing: " + str(missing_q) + ". Finish Section 6 first."
    )

print(f"Reconstructed {len(mitigated_answers_df)} controlled answers.")
print(f"Answers that still need human review: "
      f"{int(mitigated_answers_df['needs_human_review'].sum())}")

# ---- Save Part 2 outputs ----
mitigation_trace_csv_df = mitigation_trace_df.copy()
mitigation_trace_csv_df["attempts"] = mitigation_trace_csv_df["attempts"].apply(json.dumps)
mitigation_trace_csv_df["final_evidence_text"] = ""
mitigation_trace_csv_df.to_csv(MITIGATION_TRACE_CSV_PATH, index=False)

mitigated_answers_df.to_csv(MITIGATED_ANSWERS_CSV_PATH, index=False)
with open(MITIGATED_ANSWERS_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, r in mitigated_answers_df.iterrows():
        f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

human_review_df.to_csv(HUMAN_REVIEW_LOG_CSV_PATH, index=False)
with open(HUMAN_REVIEW_LOG_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, r in human_review_df.iterrows():
        f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

print(f"Saved: {MITIGATION_TRACE_CSV_PATH}")
print(f"Saved: {MITIGATED_ANSWERS_CSV_PATH}")
print(f"Saved: {HUMAN_REVIEW_LOG_CSV_PATH}")

# ---- MITIGATION SUMMARY (AFTER the hallucination control) ----
print("\n========== MITIGATION SUMMARY (AFTER CONTROL) ==========")
print(f"Total claims                     : {len(mitigation_trace_df)}")
n_keep = int((mitigation_trace_df["final_action"] == "keep").sum())
n_correct = int((mitigation_trace_df["final_action"].isin(["correct", "correct_after_reretrieval"])).sum())
n_ground = int((mitigation_trace_df["final_action"] == "ground").sum())
n_reretrieve = int((mitigation_trace_df["final_action"] == "keep_after_reretrieval").sum())
n_review = int((mitigation_trace_df["final_status"] == "UNRESOLVED_HUMAN_REVIEW").sum())
print(f"  originally supported and kept  : {n_keep}")
print(f"  corrected (rewritten, NLI-ok)  : {n_correct}")
print(f"  grounded to evidence (NLI-ok)  : {n_ground}")
print(f"  recovered via re-retrieval     : {n_reretrieve}")
print(f"  sent to human review (removed) : {n_review}")
print(f"Mitigation success rate (resolved / total): "
      f"{(mitigation_trace_df['final_status'].str.startswith('RESOLVED')).mean():.2%}")
print(f"Unresolved rate (removed / total)         : {n_review / len(mitigation_trace_df):.2%}")
print(f"\nNOTE: unresolved claims are REMOVED from the emitted answer, not counted as corrected.")


Reconstructed 100 controlled answers.
Answers that still need human review: 32
Saved: d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\mitigation_trace.csv
Saved: d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\mitigated_rag_answers.csv
Saved: d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\human_expert_review_log.csv

========== MITIGATION SUMMARY (AFTER CONTROL) ==========
Total claims                     : 644
  originally supported and kept  : 465
  corrected (rewritten, NLI-ok)  : 18
  grounded to evidence (NLI-ok)  : 83
  recovered via re-retrieval     : 9
  sent to human review (removed) : 69
Mitigation success rate (resolved / total): 89.29%
Unresolved rate (removed / total)         : 10.71%

NOTE: unresolved claims are REMOVED from the emitted answer, not counted as corrected.


---
# 8. Ablation Evaluation (WITHOUT vs WITH the control module)

In [11]:
# Section 8a - Evaluation helpers (same implementation for both conditions).

# ---- ROUGE ----
try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("Warning: rouge_score not installed; ROUGE metrics will be skipped.")

_rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True) if HAS_ROUGE else None


def _bertscore_f1(preds, refs, device="cpu"):
    """BERTScore F1 with the SAME configuration as Notebook 1 (allenai/scibert_scivocab_uncased)."""
    from collections import defaultdict
    from bert_score.utils import get_model, get_tokenizer, bert_cos_score_idf

    tokenizer = get_tokenizer(BERTSCORE_MODEL, use_fast=False)
    tokenizer.model_max_length = 512
    model = get_model(BERTSCORE_MODEL, 8, False).to(device)
    idf_dict = defaultdict(lambda: 1.0)
    idf_dict[tokenizer.sep_token_id] = 0
    idf_dict[tokenizer.cls_token_id] = 0
    all_preds = bert_cos_score_idf(
        model, refs, preds, tokenizer, idf_dict, verbose=False, batch_size=64, device=device
    )
    if isinstance(all_preds, tuple):
        F = all_preds[2]
    else:
        F = all_preds[:, 2]
    f1_arr = F.numpy()
    return f1_arr, float(f1_arr.mean())


def _meteor_score_safe(ref_tokens, pred_tokens):
    try:
        return meteor_score([ref_tokens], pred_tokens)
    except Exception:
        try:
            return meteor_score(ref_tokens, pred_tokens)
        except Exception:
            return np.nan


def semantic_metrics(preds, refs):
    """ROUGE-1, ROUGE-L, BLEU, BERTScore-F1 (scibert), METEOR.

    Returns (per_question_df, summary_dict). Empty strings are replaced with "." so the
    metric functions never crash (same guard as Notebook 1).
    """
    preds = [str(p) if p is not None and str(p).strip() else "." for p in preds]
    refs = [str(r) if r is not None and str(r).strip() else "." for r in refs]

    per_q = pd.DataFrame(index=range(len(preds)))
    if HAS_ROUGE:
        r1, rl = [], []
        for p, r in zip(preds, refs):
            if p != "." and r != ".":
                s = _rouge.score(r, p)
                r1.append(s["rouge1"].fmeasure)
                rl.append(s["rougeL"].fmeasure)
            else:
                r1.append(np.nan)
                rl.append(np.nan)
        per_q["rouge1_f1"] = r1
        per_q["rougeL_f1"] = rl

    smooth = SmoothingFunction().method1
    pred_tok = [p.split() for p in preds]
    ref_tok = [[r.split()] for r in refs]
    per_q["bleu"] = [
        sentence_bleu(ref_tok[i], pred_tok[i], weights=(0.25, 0.25, 0.25, 0.25),
                      smoothing_function=smooth)
        for i in range(len(pred_tok))
    ]

    try:
        import sacrebleu
        bleu_corpus = sacrebleu.corpus_bleu(preds, [refs]).score
    except Exception:
        bleu_corpus = corpus_bleu(ref_tok, pred_tok, weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=smooth)

    bert_f1 = np.nan
    try:
        bert_arr, bert_f1 = _bertscore_f1(preds, refs, device=device)
        if len(bert_arr) == len(per_q):
            per_q["bertscore_f1"] = bert_arr
        else:
            per_q["bertscore_f1"] = np.nan
    except Exception as exc:
        print("Warning: BERTScore skipped:", exc)
        per_q["bertscore_f1"] = np.nan

    meteor = np.nan
    if HAS_METEOR:
        vals = [
            _meteor_score_safe(ref_tok[i][0], pred_tok[i])
            for i in range(len(pred_tok))
            if pred_tok[i] and ref_tok[i][0]
        ]
        if vals:
            meteor = float(np.nanmean(vals))

    summary = {
        "n": len(preds),
        "rouge1_f1": float(np.nanmean(per_q["rouge1_f1"])) if HAS_ROUGE else np.nan,
        "rougeL_f1": float(np.nanmean(per_q["rougeL_f1"])) if HAS_ROUGE else np.nan,
        "bleu": float(bleu_corpus),
        "bertscore_f1": bert_f1,
        "meteor": meteor,
    }
    return per_q, summary


def nli_metrics_for_claims(claims_for_eval):
    """Claim-level NLI metrics from a list of verification labels."""
    n = len(claims_for_eval)
    if n == 0:
        return {"total_claims": 0, "supported_rate": 0.0, "contradicted_rate": 0.0,
                "missing_evidence_rate": 0.0, "unsupported_rate": 0.0}
    supported = sum(1 for l in claims_for_eval if l == "SUPPORTED")
    contradicted = sum(1 for l in claims_for_eval if l == "CONTRADICTED")
    missing = sum(1 for l in claims_for_eval if l == "NOT_ENOUGH_EVIDENCE")
    return {
        "total_claims": n,
        "supported_rate": supported / n,
        "contradicted_rate": contradicted / n,
        "missing_evidence_rate": missing / n,
        "unsupported_rate": (contradicted + missing) / n,
    }


# ---- Evidence pool per question (kept chunks + chunks re-retrieved during mitigation) ----
def build_evidence_pool_by_question():
    pool = {}
    for qid in mitigated_answers_df["question_id"].unique():
        kept = [dict(c) for c in df.loc[qid, "kept_chunks_parsed"]]
        seen = {c.get("chunk_id") for c in kept}
        for _, t in mitigation_trace_df[mitigation_trace_df["question_id"] == qid].iterrows():
            for attempt in (t.get("attempts") or []):
                for chunk in attempt.get("new_chunks", []):
                    if chunk.get("chunk_id") and chunk["chunk_id"] not in seen:
                        seen.add(chunk["chunk_id"])
                        kept.append({
                            "chunk_id": chunk["chunk_id"],
                            "pmid": chunk.get("pmid"),
                            "chunk_text": chunk.get("chunk_text"),
                        })
        pool[qid] = kept
    return pool


evidence_pool_by_question = build_evidence_pool_by_question()
print(f"Evidence pools built for {len(evidence_pool_by_question)} questions.")
pool_sizes = [len(v) for v in evidence_pool_by_question.values()]
print(f"Evidence pool size: mean {np.mean(pool_sizes):.1f}, max {max(pool_sizes)}, "
      f"min {min(pool_sizes)}")


# ---- Sentence-level faithfulness (identical to Notebook 1's scorer) ----
_BOILERPLATE_PATTERNS = [
    r"\bbased on the (retrieved )?evidence\b[,:]?",
    r"\baccording to the (retrieved )?evidence\b[,:]?",
    r"\bthe retrieved evidence (suggests|indicates|shows|supports)\b",
    r"\bthe evidence suggests\b",
    r"\bin summary\b[,:]?",
    r"\boverall\b[,:]?",
    r"\bin conclusion\b[,:]?",
    r"\bas a result\b[,:]?",
]
_DISCLAIMER_PATTERNS = [
    r"\b(i am not a doctor|not (intended|meant) as medical advice|this is not medical advice|"
    r"please consult|always seek (the advice|medical advice)|if you have any concerns|"
    r"seek professional (medical|healthcare) advice)\b",
    r"\bthis information (should not|is not intended to)\b",
    r"\bpersonal medical advice\b",
]
_GENERIC_PATTERNS = [
    r"\bit (is|would be) (important|essential|crucial|worth) (to|noting|noting that)\b",
    r"\bit's? worth noting\b",
    r"\bfurther (research|studies) (are|is) (needed|warranted|required)\b",
    r"\bmore (research|studies|work) (is|are) (needed|warranted|required)\b",
    r"\badditional (studies|research) (are|is) needed\b",
    r"\blarger (studies|trials) (are|is) needed\b",
    r"\bfuture (studies|research) (should|are|will)\b",
    r"\bthese findings (should be interpreted|highlight the)\b",
    r"\bcaution (is|should be) (warranted|exercised)\b",
    r"\bthe (study|trial) (was|has|had|is) (conducted|performed|limited)\b",
    r"\bthe (results|findings) (should be|may (not be)) (interpreted|generalized|generalisable|generalizable)\b",
    r"\bgeneralizab\w* to other\b",
]
_STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "for", "to", "and", "or", "is", "are",
    "was", "were", "with", "as", "by", "at", "from", "that", "this", "these",
    "those", "be", "been", "being", "it", "its", "than", "then", "do", "does",
    "did", "can", "could", "should", "would", "will", "may", "might", "have",
    "has", "had", "not", "no", "vs", "versus", "between", "which", "what",
    "who", "how", "why", "when", "study", "studies", "patient", "patients",
}
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)


def _keywords(text):
    text = (text or "").lower().translate(_PUNCT_TABLE)
    return {t for t in text.split() if len(t) > 3 and t not in _STOPWORDS}


def _has_pattern(text, patterns):
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def _strip_citations(text):
    return re.sub(r"\(?\s*Evidence\s*\d+\s*\)?", " ", text, flags=re.IGNORECASE)


def _nli_probs(premise, hypothesis):
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt",
                           truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {nli_id2label[i]: float(probs[i]) for i in range(len(probs))}


def sentence_faithfulness(sentences, evidence_texts):
    """Sentence-level faithfulness (Notebook 1 scorer): hybrid NLI + lexical overlap.

    Returns (records, faithfulness, unsupported_sentence_rate, counts).
    """
    records = []
    nli_memo = {}
    for sentence in sentences:
        if _has_pattern(sentence, _DISCLAIMER_PATTERNS) or _has_pattern(sentence, _GENERIC_PATTERNS):
            records.append({"sentence": sentence, "label": "non_factual_excluded", "score": None,
                            "p_entailment": None, "p_contradiction": None, "lexical_overlap": None})
            continue
        if len(evidence_texts) == 0:
            records.append({"sentence": sentence, "label": "unsupported", "score": 0.0,
                            "p_entailment": None, "p_contradiction": None, "lexical_overlap": 0.0})
            continue

        hyp = _strip_citations(sentence)
        skw = _keywords(hyp)
        best_ent, best_con, best_ov = 0.0, 0.0, 0.0
        for et in evidence_texts:
            key = (et, hyp)
            lp = nli_memo.get(key)
            if lp is None:
                lp = _nli_probs(et, hyp)
                nli_memo[key] = lp
            ent, con = lp.get("entailment", 0.0), lp.get("contradiction", 0.0)
            best_ent = max(best_ent, ent)
            best_con = max(best_con, con)
            ekw = _keywords(et)
            ov = (len(skw & ekw) / len(skw)) if skw else 0.0
            best_ov = max(best_ov, ov)

        combined = ENTAIL_WEIGHT * best_ent + OVERLAP_WEIGHT * best_ov
        if combined >= SUPPORTED_COMBINED_THRESHOLD and best_con < best_ent + 0.10:
            label, score = "supported", 1.0
        elif best_con >= CONTRADICT_PROB_THRESHOLD and best_con > combined:
            label, score = "unsupported", 0.0
        else:
            label, score = "partially_supported", 0.5

        records.append({"sentence": sentence, "label": label, "score": score,
                        "p_entailment": best_ent, "p_contradiction": best_con,
                        "lexical_overlap": best_ov})

    factual = [r for r in records if r["label"] != "non_factual_excluded"]
    n = len(factual)
    n_sup = sum(1 for r in factual if r["label"] == "supported")
    n_part = sum(1 for r in factual if r["label"] == "partially_supported")
    n_unsup = sum(1 for r in factual if r["label"] == "unsupported")

    faithfulness = ((n_sup + 0.5 * n_part) / n) if n > 0 else np.nan
    unsupported_rate = (n_unsup / n) if n > 0 else np.nan
    counts = {"n_factual": n, "n_supported": n_sup,
              "n_partially_supported": n_part, "n_unsupported": n_unsup}
    return records, faithfulness, unsupported_rate, counts


print("Evaluation helpers ready.")


Evidence pools built for 100 questions.
Evidence pool size: mean 9.6, max 47, min 1
Evaluation helpers ready.


In [12]:
# Section 8b - VALIDATION GATE: refuse an unfair / incorrect ablation.
def validate_ablation_ready():
    problems = []

    if len(df) != REQUIRED_N:
        problems.append(f"Baseline (without control) has {len(df)} answers (expected {REQUIRED_N}).")
    if len(mitigated_answers_df) != REQUIRED_N:
        problems.append(f"Controlled (with control) has {len(mitigated_answers_df)} answers (expected {REQUIRED_N}).")
    if len(claims_df) == 0:
        problems.append("No verified claims (Section 5 produced nothing).")
    if len(mitigation_trace_df) == 0:
        problems.append("No mitigation trace (Section 6 produced nothing).")

    base_q = df["question"].astype(str).tolist()
    contr_q = mitigated_answers_df["question"].astype(str).tolist()

    if len(base_q) != len(set(base_q)):
        problems.append("Baseline contains duplicated questions.")
    if len(contr_q) != len(set(contr_q)):
        problems.append("Controlled contains duplicated questions.")
    if set(base_q) != set(contr_q):
        problems.append("Baseline and controlled question sets differ - not the same 100 questions.")

    b = df.set_index("question")
    c = mitigated_answers_df.set_index("question")
    ref_diff = [q for q in base_q if q in c.index and b.loc[q, "gold_long_answer"] != c.loc[q, "gold_long_answer"]]
    pmid_diff = [q for q in base_q if q in c.index and str(b.loc[q, "gold_pmid"]) != str(c.loc[q, "gold_pmid"])]
    if ref_diff:
        problems.append(f"{len(ref_diff)} questions have DIFFERENT gold references between conditions.")
    if pmid_diff:
        problems.append(f"{len(pmid_diff)} questions have DIFFERENT gold PMIDs between conditions.")

    traced = set(mitigation_trace_df["claim_id"])
    unprocessed = [cid for cid in claims_df["claim_id"] if cid not in traced]
    if unprocessed:
        problems.append(f"{len(unprocessed)} claims are still pending (no mitigation result).")

    q_with_claims = set(claims_df["question_id"])
    q_without = [qid for qid in df.index if qid not in q_with_claims]
    if q_without:
        problems.append(f"{len(q_without)} answers have no claims (decomposition/detection missing).")

    if problems:
        print("========== ABLATION DATA VALIDATION FAILED ==========")
        for p in problems:
            print("  [x]", p)
        print("The final ablation was NOT computed to avoid an unfair or incorrect comparison.")
        print("Fix the issues (usually: re-run the earlier sections to finish pending Gemini work),")
        print("then re-run this section.")
        raise RuntimeError("Ablation validation failed: the comparison would not be 100-vs-100 "
                           "on the same questions.")
    print("========== ABLATION DATA VALIDATION PASSED ==========")
    print(f"Baseline   : {len(df)} answers | Controlled: {len(mitigated_answers_df)} answers")
    print(f"Same {REQUIRED_N} questions, same gold references and PMIDs, no duplicates, "
          f"no pending claims. Proceeding with the ablation.")


validate_ablation_ready()


========== ABLATION DATA VALIDATION PASSED ==========
Baseline   : 100 answers | Controlled: 100 answers
Same 100 questions, same gold references and PMIDs, no duplicates, no pending claims. Proceeding with the ablation.


In [13]:
# Section 8c - Ablation run A: WITHOUT the hallucination control (baseline).

# Claim-level rates come from the Section 5 detection labels (before correction).
baseline_claim_labels = claims_df["verification_label"].tolist()
baseline_nli = nli_metrics_for_claims(baseline_claim_labels)

# Answer-quality metrics on the SAME reference answers (gold_long_answer).
baseline_per_q, baseline_summary = semantic_metrics(
    df["answer_text"].tolist(), df["gold_long_answer"].tolist()
)

# Faithfulness: the sentence-level faithfulness computed by Notebook 1 (same scorer).
baseline_faithfulness = float(df["faithfulness"].mean())
baseline_risk = float(answer_summary_df["hallucination_risk"].mean())

# FActScore = fraction of supported claims per answer (averaged over answers).
baseline_factscore_per_q = claims_df.groupby("question_id").apply(
    lambda g: (g["verification_label"] == "SUPPORTED").mean()
).mean()

# Answer-level Hallucination Rate = fraction of answers with >= 1 unsupported claim.
baseline_answer_hr = float(
    claims_df.groupby("question_id").apply(
        lambda g: ((g["verification_label"] == "CONTRADICTED") |
                   (g["verification_label"] == "NOT_ENOUGH_EVIDENCE")).any()
    ).mean()
)

baseline_metrics = {
    "n_answers": len(df),
    "total_claims": baseline_nli["total_claims"],
    "supported_rate": baseline_nli["supported_rate"],
    "contradicted_rate": baseline_nli["contradicted_rate"],
    "missing_evidence_rate": baseline_nli["missing_evidence_rate"],
    "unsupported_rate": baseline_nli["unsupported_rate"],
    "hallucination_risk": baseline_risk,
    "rouge1_f1": baseline_summary["rouge1_f1"],
    "rougeL_f1": baseline_summary["rougeL_f1"],
    "bleu": baseline_summary["bleu"],
    "bertscore_f1": baseline_summary["bertscore_f1"],
    "meteor": baseline_summary["meteor"],
    "faithfulness": baseline_faithfulness,
    "factscore": baseline_factscore_per_q,
    "answer_level_hr": baseline_answer_hr,
}

print("=== WITHOUT hallucination control (baseline) ===")
for k, v in baseline_metrics.items():
    if v is not None and not (isinstance(v, float) and np.isnan(v)):
        if isinstance(v, float):
            print(f"  {k:24s}: {v:.4f}")
        else:
            print(f"  {k:24s}: {v}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== WITHOUT hallucination control (baseline) ===
  n_answers               : 100
  total_claims            : 644
  supported_rate          : 0.7220
  contradicted_rate       : 0.0357
  missing_evidence_rate   : 0.2422
  unsupported_rate        : 0.2780
  hallucination_risk      : 0.2816
  rouge1_f1               : 0.3204
  rougeL_f1               : 0.2116
  bleu                    : 5.5950
  bertscore_f1            : 0.6394
  meteor                  : 0.2525
  faithfulness            : 0.8613
  factscore               : 0.7184
  answer_level_hr         : 0.6100


C:\Users\YOGA\AppData\Local\Temp\ipykernel_13180\3968751843.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseline_factscore_per_q = claims_df.groupby("question_id").apply(
C:\Users\YOGA\AppData\Local\Temp\ipykernel_13180\3968751843.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  claims_df.groupby("question_id").apply(


In [15]:
# Section 8d - Ablation run B: WITH the hallucination control (mitigated answers).

# Emitted claims (resolved ones only) are re-verified against the answer's evidence pool.
controlled_claim_labels = []
for qid in mitigated_answers_df["question_id"].unique():
    pool = evidence_pool_by_question.get(qid, [])
    for _, t in mitigation_trace_df[mitigation_trace_df["question_id"] == qid].iterrows():
        if t["final_status"] == "UNRESOLVED_HUMAN_REVIEW":
            continue  # escalated content is not emitted in the controlled answer
        recheck = verify_claim_against_new_evidence(t["final_claim"], pool)
        controlled_claim_labels.append(recheck["new_verification_label"])

controlled_nli = nli_metrics_for_claims(controlled_claim_labels)

# Answer-quality metrics on the SAME gold references.
controlled_per_q, controlled_summary = semantic_metrics(
    mitigated_answers_df["mitigated_answer"].tolist(),
    mitigated_answers_df["gold_long_answer"].tolist(),
)

# Sentence-level faithfulness of the controlled answers vs their evidence pool.
controlled_faith_scores = []
for _, r in mitigated_answers_df.iterrows():
    qid = r["question_id"]
    pool = evidence_pool_by_question.get(qid, [])
    sents = [s for s in sent_tokenize(str(r["mitigated_answer"])) if len(s.split()) >= 2]
    records, faithfulness, _, _ = sentence_faithfulness(sents, [c["chunk_text"] for c in pool])
    controlled_faith_scores.append(faithfulness)
controlled_faithfulness = float(np.nanmean(controlled_faith_scores))

# Unresolved share = claims escalated to human review (removed from the answer).
unresolved_rate = float(mitigated_answers_df["unresolved_rate"].mean())

# FActScore for controlled: among emitted (resolved) claims, fraction verified SUPPORTED.
controlled_factscore_per_q = []
for qid in mitigated_answers_df["question_id"].unique():
    emitted = [l for l, t in zip(controlled_claim_labels, [
        t for _, t in mitigation_trace_df[mitigation_trace_df["question_id"] == qid].iterrows()
        if t["final_status"] != "UNRESOLVED_HUMAN_REVIEW"
    ]) if True]
    # Count from the re-verified labels of emitted claims only
controlled_factscore = float(np.mean([
    l == "SUPPORTED" for l in controlled_claim_labels
])) if controlled_claim_labels else 0.0

# Answer-level HR for controlled: among emitted claims, fraction of answers with >=1 unsupported.
controlled_answer_hrs = []
for qid in mitigated_answers_df["question_id"].unique():
    q_emitted_labels = []
    for _, t in mitigation_trace_df[mitigation_trace_df["question_id"] == qid].iterrows():
        if t["final_status"] == "UNRESOLVED_HUMAN_REVIEW":
            continue
        recheck = verify_claim_against_new_evidence(t["final_claim"], evidence_pool_by_question.get(qid, []))
        q_emitted_labels.append(recheck["new_verification_label"])
    has_unsupported = any(l in ("CONTRADICTED", "NOT_ENOUGH_EVIDENCE") for l in q_emitted_labels)
    controlled_answer_hrs.append(has_unsupported)
controlled_answer_hr = float(np.mean(controlled_answer_hrs)) if controlled_answer_hrs else 0.0

controlled_metrics = {
    "n_answers": len(mitigated_answers_df),
    "total_claims": controlled_nli["total_claims"],
    "supported_rate": controlled_nli["supported_rate"],
    "contradicted_rate": controlled_nli["contradicted_rate"],
    "missing_evidence_rate": controlled_nli["missing_evidence_rate"],
    "unsupported_rate": controlled_nli["unsupported_rate"],
    "unresolved_rate": unresolved_rate,
    "rouge1_f1": controlled_summary["rouge1_f1"],
    "rougeL_f1": controlled_summary["rougeL_f1"],
    "bleu": controlled_summary["bleu"],
    "bertscore_f1": controlled_summary["bertscore_f1"],
    "meteor": controlled_summary["meteor"],
    "faithfulness": controlled_faithfulness,
    "factscore": controlled_factscore,
    "answer_level_hr": controlled_answer_hr,
}

print("=== WITH hallucination control (mitigated) ===")
for k, v in controlled_metrics.items():
    if v is not None and not (isinstance(v, float) and np.isnan(v)):
        if isinstance(v, float):
            print(f"  {k:24s}: {v:.4f}")
        else:
            print(f"  {k:24s}: {v}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== WITH hallucination control (mitigated) ===
  n_answers               : 100
  total_claims            : 575
  supported_rate          : 0.9600
  contradicted_rate       : 0.0070
  missing_evidence_rate   : 0.0330
  unsupported_rate        : 0.0400
  unresolved_rate         : 0.0998
  rouge1_f1               : 0.2408
  rougeL_f1               : 0.1639
  bleu                    : 3.1744
  bertscore_f1            : 0.5901
  meteor                  : 0.2172
  faithfulness            : 0.9730
  factscore               : 0.9600
  answer_level_hr         : 0.0800


---
# 9. Final Results

In [16]:
# Section 9 - Final tables.

def _fmt(v, pct=False):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "-"
    if pct:
        return f"{100.0 * v:.2f}%"
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)


# ---------- Table 1: ablation comparison ----------
# Mitigation outcomes used to enrich the table.
n_removed = int((mitigation_trace_df["final_status"] == "UNRESOLVED_HUMAN_REVIEW").sum())
n_corrected = int(mitigation_trace_df["final_action"].isin(
    ["correct", "correct_after_reretrieval", "ground"]).sum())
n_reretrieved = int((mitigation_trace_df["final_action"] == "keep_after_reretrieval").sum())
n_recovered = n_corrected + n_reretrieved

rows = [
    ("Answers evaluated", baseline_metrics["n_answers"], controlled_metrics["n_answers"], False),
    ("Total claims analyzed (emitted)", baseline_metrics["total_claims"], controlled_metrics["total_claims"], False),
    ("Supported claim rate", baseline_metrics["supported_rate"], controlled_metrics["supported_rate"], True),
    ("Contradicted claim rate", baseline_metrics["contradicted_rate"], controlled_metrics["contradicted_rate"], True),
    ("Unsupported claim rate (HR)", baseline_metrics["unsupported_rate"], controlled_metrics["unsupported_rate"], True),
    ("Faithfulness", baseline_metrics["faithfulness"], controlled_metrics["faithfulness"], False),
    ("FActScore", baseline_metrics["factscore"], controlled_metrics["factscore"], False),
    ("Answer-level HR", baseline_metrics["answer_level_hr"], controlled_metrics["answer_level_hr"], True),
    ("Claims recovered automatically", 0, n_recovered, False),
    ("Claims escalated to human review", 0, n_removed, False),
]

comparison_rows = []
for name, b, c, is_pct in rows:
    delta = None
    if isinstance(b, (int, float, np.integer)) and isinstance(c, (int, float, np.integer)):
        bv = b if not (isinstance(b, float) and np.isnan(b)) else None
        cv = c if not (isinstance(c, float) and np.isnan(c)) else None
        if bv is not None and cv is not None:
            delta = cv - bv
    if delta is None:
        delta_str = "-"
    elif is_pct:
        delta_str = f"{delta:+.2f} pp"
    elif isinstance(delta, (int, np.integer)):
        delta_str = f"{delta:+d}"
    else:
        delta_str = f"{delta:+.4f}"
    comparison_rows.append({
        "metric": name,
        "without_control": _fmt(b, is_pct),
        "with_control": _fmt(c, is_pct),
        "delta": delta_str,
    })

comparison_df = pd.DataFrame(comparison_rows)

md_lines = [
    "| Metric | Without control | With control | Delta |",
    "|---|---|---|---|",
    "| **HR reduction** | **{0:.2f}%** | **{1:.2f}%** | **{2:+.2f} pp** |".format(
        100.0 * baseline_metrics["unsupported_rate"],
        100.0 * controlled_metrics["unsupported_rate"],
        100.0 * (controlled_metrics["unsupported_rate"] - baseline_metrics["unsupported_rate"]),
    ),
]
for _, r in comparison_df.iterrows():
    md_lines.append(f"| {r['metric']} | {r['without_control']} | {r['with_control']} | {r['delta']} |")
display(Markdown("\n".join(md_lines)))


# ---------- Table 2: mitigation outcomes (mutually exclusive categories) ----------
keep_count = int((mitigation_trace_df["final_action"] == "keep").sum())
corrected_count = int(mitigation_trace_df["final_action"].isin(
    ["correct", "correct_after_reretrieval", "ground"]).sum())
reretrieved_count = int((mitigation_trace_df["final_action"] == "keep_after_reretrieval").sum())
human_review_count = int((mitigation_trace_df["final_action"] == "human_review").sum())
total_outcome = keep_count + corrected_count + reretrieved_count + human_review_count

outcome_rows = [
    ("Originally supported and kept", keep_count),
    ("Corrected (rewritten from evidence, NLI-confirmed)", corrected_count),
    ("Recovered via re-retrieval (kept with new evidence)", reretrieved_count),
    ("Sent to human review (removed from answer)", human_review_count),
    ("Total claims processed", total_outcome),
]
mitigation_outcome_df = pd.DataFrame(outcome_rows, columns=["outcome", "count"])
mitigation_outcome_df["share"] = (
    (mitigation_outcome_df["count"] / total_outcome * 100).round(2) if total_outcome else 0.0
)

md2_lines = ["| Hallucination Control Outcome | Count | Share |", "|---|---|---|"]
for _, r in mitigation_outcome_df.iterrows():
    md2_lines.append(f"| {r['outcome']} | {r['count']} | {r['share']:.2f}% |")
display(Markdown("\n".join(md2_lines)))

print("Note: the four outcome categories are mutually exclusive and sum to the total number of\n"
      "claims. 'Corrected' groups evidence-authoritative rewrites (correct / correct_after_\n"
      "reretrieval / ground), all accepted only after NLI confirmation against the evidence.")


| Metric | Without control | With control | Delta |
|---|---|---|---|
| **HR reduction** | **27.80%** | **4.00%** | **-23.80 pp** |
| Answers evaluated | 100 | 100 | +0 |
| Total claims analyzed (emitted) | 644 | 575 | -69 |
| Supported claim rate | 72.20% | 96.00% | +0.24 pp |
| Contradicted claim rate | 3.57% | 0.70% | -0.03 pp |
| Unsupported claim rate (HR) | 27.80% | 4.00% | -0.24 pp |
| Faithfulness | 0.8613 | 0.9730 | +0.1117 |
| FActScore | 0.7184 | 0.9600 | +0.2416 |
| Answer-level HR | 61.00% | 8.00% | -0.53 pp |
| Claims recovered automatically | 0 | 110 | +110 |
| Claims escalated to human review | 0 | 69 | +69 |

| Hallucination Control Outcome | Count | Share |
|---|---|---|
| Originally supported and kept | 465 | 72.20% |
| Corrected (rewritten from evidence, NLI-confirmed) | 101 | 15.68% |
| Recovered via re-retrieval (kept with new evidence) | 9 | 1.40% |
| Sent to human review (removed from answer) | 69 | 10.71% |
| Total claims processed | 644 | 100.00% |

Note: the four outcome categories are mutually exclusive and sum to the total number of
claims. 'Corrected' groups evidence-authoritative rewrites (correct / correct_after_
reretrieval / ground), all accepted only after NLI confirmation against the evidence.


---
# 9b. Visualization

In [17]:
# Section 9b - Visualization: detection breakdown and mitigation outcomes.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

COLORS = {
    "supported": "#2ecc71",
    "contradicted": "#e74c3c",
    "nee": "#f39c12",
    "recovered": "#3498db",
    "human_review": "#95a5a6",
    "baseline": "#7f8c8d",
    "controlled": "#2980b9",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# ---- Panel A: Detection breakdown (before control) ----
n_supported = int((claims_df["verification_label"] == "SUPPORTED").sum())
n_contradicted = int((claims_df["verification_label"] == "CONTRADICTED").sum())
n_nee = int((claims_df["verification_label"] == "NOT_ENOUGH_EVIDENCE").sum())
total = n_supported + n_contradicted + n_nee

det_labels = ["Supported", "Contradicted", "NEE"]
det_counts = [n_supported, n_contradicted, n_nee]
det_colors = [COLORS["supported"], COLORS["contradicted"], COLORS["nee"]]
det_pcts = [100.0 * c / total for c in det_counts]

bars_a = axes[0].bar(det_labels, det_pcts, color=det_colors, edgecolor="white", linewidth=1.2)
for bar, pct, cnt in zip(bars_a, det_pcts, det_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                 f"{pct:.1f}%\n({cnt})", ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].set_ylabel("Share of claims (%)", fontsize=11)
axes[0].set_title("A. Detection Breakdown (Before Control)", fontsize=12, fontweight="bold")
axes[0].set_ylim(0, max(det_pcts) * 1.25)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# ---- Panel B: Mitigation outcomes ----
n_keep = int((mitigation_trace_df["final_action"] == "keep").sum())
n_corrected = int(mitigation_trace_df["final_action"].isin(
    ["correct", "correct_after_reretrieval", "ground"]).sum())
n_reretrieved = int((mitigation_trace_df["final_action"] == "keep_after_reretrieval").sum())
n_human = int((mitigation_trace_df["final_action"] == "human_review").sum())

mit_labels = ["Supported\n(kept)", "Corrected/\nGrounded", "Re-retrieved\n(kept)", "Human\nReview"]
mit_counts = [n_keep, n_corrected, n_reretrieved, n_human]
mit_colors = [COLORS["supported"], COLORS["recovered"], "#1abc9c", COLORS["human_review"]]
mit_pcts = [100.0 * c / total for c in mit_counts]

bars_b = axes[1].bar(mit_labels, mit_pcts, color=mit_colors, edgecolor="white", linewidth=1.2)
for bar, pct, cnt in zip(bars_b, mit_pcts, mit_counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                 f"{pct:.1f}%\n({cnt})", ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[1].set_ylabel("Share of claims (%)", fontsize=11)
axes[1].set_title("B. Mitigation Outcomes", fontsize=12, fontweight="bold")
axes[1].set_ylim(0, max(mit_pcts) * 1.25)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

# ---- Panel C: Ablation comparison (7 metrics) ----
metric_names = ["Supported\nRate", "Contradicted\nRate", "Unsupported\nRate",
                "Faithfulness", "FActScore", "HR", "Answer-level\nHR"]
baseline_vals = [
    baseline_metrics["supported_rate"],
    baseline_metrics["contradicted_rate"],
    baseline_metrics["unsupported_rate"],
    baseline_metrics["faithfulness"],
    baseline_metrics["factscore"],
    baseline_metrics["unsupported_rate"],
    baseline_metrics["answer_level_hr"],
]
controlled_vals = [
    controlled_metrics["supported_rate"],
    controlled_metrics["contradicted_rate"],
    controlled_metrics["unsupported_rate"],
    controlled_metrics["faithfulness"],
    controlled_metrics["factscore"],
    controlled_metrics["unsupported_rate"],
    controlled_metrics["answer_level_hr"],
]

x = np.arange(len(metric_names))
w = 0.35
bars_bl = axes[2].bar(x - w/2, [v * 100 for v in baseline_vals], w,
                      label="Without Control", color=COLORS["baseline"], edgecolor="white")
bars_co = axes[2].bar(x + w/2, [v * 100 for v in controlled_vals], w,
                      label="With Control", color=COLORS["controlled"], edgecolor="white")
axes[2].set_xticks(x)
axes[2].set_xticklabels(metric_names, fontsize=9)
axes[2].set_ylabel("Score (%)", fontsize=11)
axes[2].set_title("C. Ablation Comparison", fontsize=12, fontweight="bold")
axes[2].legend(fontsize=9, loc="upper left")
axes[2].set_ylim(0, 110)
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[2].spines["top"].set_visible(False)
axes[2].spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(HALL_CONTROL_DIR / "ablation_visualization.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {HALL_CONTROL_DIR / 'ablation_visualization.png'}")


Saved: d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\ablation_visualization.png


C:\Users\YOGA\AppData\Local\Temp\ipykernel_13180\1037719730.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
# 10. Save Outputs

Saves every table produced by this notebook (ablation comparison, mitigation outcomes, per-question
metrics for both conditions) into `data/outputs/hallucination_control/`.

In [18]:
# Section 10 - Save all final outputs.

comparison_df.to_csv(ABLATION_CSV_PATH, index=False)
mitigation_outcome_df.to_csv(MITIGATION_OUTCOME_CSV_PATH, index=False)
baseline_per_q.to_csv(BASELINE_PER_Q_METRICS_CSV, index=False)
controlled_per_q.to_csv(CONTROLLED_PER_Q_METRICS_CSV, index=False)

print("Saved outputs:")
print(f"  {ABLATION_CSV_PATH}")
print(f"  {MITIGATION_OUTCOME_CSV_PATH}")
print(f"  {BASELINE_PER_Q_METRICS_CSV}")
print(f"  {CONTROLLED_PER_Q_METRICS_CSV}")

print("\n================ FINAL ABLATION SUMMARY ================")
print(f"Questions evaluated       : {baseline_metrics['n_answers']} (baseline) vs "
      f"{controlled_metrics['n_answers']} (controlled) - the SAME {REQUIRED_N} questions.")
print(f"Supported claim rate      : {baseline_metrics['supported_rate']:.2%} -> "
      f"{controlled_metrics['supported_rate']:.2%}")
print(f"Contradicted claim rate   : {baseline_metrics['contradicted_rate']:.2%} -> "
      f"{controlled_metrics['contradicted_rate']:.2%}")
print(f"Unsupported claim rate    : {baseline_metrics['unsupported_rate']:.2%} -> "
      f"{controlled_metrics['unsupported_rate']:.2%}")
print(f"Faithfulness              : {baseline_metrics['faithfulness']:.4f} -> "
      f"{controlled_metrics['faithfulness']:.4f}")
print(f"FActScore                 : {baseline_metrics['factscore']:.4f} -> "
      f"{controlled_metrics['factscore']:.4f}")
print(f"Hallucination Rate (HR)   : {baseline_metrics['unsupported_rate']:.2%} -> "
      f"{controlled_metrics['unsupported_rate']:.2%}")
print(f"Answer-level HR           : {baseline_metrics['answer_level_hr']:.2%} -> "
      f"{controlled_metrics['answer_level_hr']:.2%}")
print(f"Claims recovered          : {n_recovered} "
      f"(corrected/grounded: {n_corrected}, re-retrieval: {n_reretrieved})")
print(f"Claims to human review    : {n_removed}")
print("=====================================================================")
print("Notebook complete: Hallucination Control + Ablation Evaluation.")


Saved outputs:
  d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\ablation_comparison.csv
  d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\mitigation_outcome.csv
  d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\ablation_baseline_per_question_metrics.csv
  d:\School\ENSI\Mémoire de recherche\el khedma\code\data\outputs\hallucination_control\ablation_controlled_per_question_metrics.csv

================ FINAL ABLATION SUMMARY ================
Questions evaluated       : 100 (baseline) vs 100 (controlled) - the SAME 100 questions.
Supported claim rate      : 72.20% -> 96.00%
Contradicted claim rate   : 3.57% -> 0.70%
Unsupported claim rate    : 27.80% -> 4.00%
Faithfulness              : 0.8613 -> 0.9730
FActScore                 : 0.7184 -> 0.9600
Hallucination Rate (HR)   : 27.80% -> 4.00%
Answer-level HR           : 61.00% -> 8.00%
Claims recovered          : 110 (correcte